In [8]:
import os
from google.cloud import storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"

def delete_sft_checkpoints(local_dir="checkpoints"):
    """
    Delete all local checkpoint files that start with 'sft_'.
    """
    if not os.path.exists(local_dir):
        return

    removed = 0
    for fname in os.listdir(local_dir):
        if fname.startswith("sft_") and fname.endswith(".pt"):
            fpath = os.path.join(local_dir, fname)
            os.remove(fpath)
            removed += 1
            print(f"🗑️ Deleted SFT checkpoint: {fname}")

    if removed == 0:
        print("ℹ️ No SFT checkpoints found to delete.")
    else:
        print(f"✅ Deleted {removed} SFT checkpoints.")

def download_latest_checkpoint(bucket_name, prefix="checkpoints", local_dir="checkpoints"):
    """
    Download only the latest NON-SFT checkpoint (.pt) file from GCS.
    """
    os.makedirs(local_dir, exist_ok=True)

    # delete local SFT checkpoints
    delete_sft_checkpoints(local_dir)

    client = storage.Client()
    bucket = client.bucket(bucket_name)

    # list blobs inside the prefix folder
    blobs = [
        b for b in bucket.list_blobs(prefix=prefix)
        if b.name.endswith(".pt") and not os.path.basename(b.name).startswith("sft_")
    ]

    
    if not blobs:
        print("❌ No NON-SFT checkpoints found in GCS.")
        return None

    # sort based on filename (model_00001.pt → latest is last)
    blobs.sort(key=lambda b: b.name)

    latest_blob = blobs[-1]
    filename = os.path.basename(latest_blob.name)
    local_path = os.path.join(local_dir, filename)

    # If file already exists AND size matches, skip
    if os.path.exists(local_path) and os.path.getsize(local_path) == latest_blob.size:
        print(f"✅ Latest checkpoint already exists locally: {filename}")
        return local_path

    print(f"⬇️ Downloading latest NON-SFT checkpoint:\n  {latest_blob.name} → {local_path}")
    latest_blob.download_to_filename(local_path)

    print(f"✅ Download complete: {local_path}")
    return local_path


# Run it
latest_ckpt = download_latest_checkpoint(bucket_name="gpt2-ultrafineweb")

ℹ️ No SFT checkpoints found to delete.
⬇️ Downloading latest NON-SFT checkpoint:
  checkpoints/model_09535.pt → checkpoints/model_09535.pt
✅ Download complete: checkpoints/model_09535.pt


In [3]:
#sft checkpoint best_sft.ppt 
from google.cloud import storage
import os

# Path where you want to save the checkpoint locally
LOCAL_PATH = "sft_best_v1.pt"

# Your GCS bucket + file
BUCKET_NAME = "gpt2-ultrafineweb"
BLOB_PATH   = "sft_checkpoints/sft_best.pt"

# Make sure your key is set
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"

def download_sft_best():
    print("🔄 Connecting to GCS...")
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    blob = bucket.blob(BLOB_PATH)

    print(f"⬇️ Downloading gs://{BUCKET_NAME}/{BLOB_PATH} → {LOCAL_PATH}")
    blob.download_to_filename(LOCAL_PATH)

    print("✅ Download complete!")

if __name__ == "__main__":
    download_sft_best()


🔄 Connecting to GCS...
⬇️ Downloading gs://gpt2-ultrafineweb/sft_checkpoints/sft_best.pt → sft_best_v1.pt
✅ Download complete!


In [4]:
from google.cloud import storage
import os

# -----------------------
# CONFIG
# -----------------------
BUCKET_NAME = "gpt2-ultrafineweb"
BLOB_PATH   = "qa-gpt2-small.jsonl"   # Path inside bucket
LOCAL_PATH  = "qa-gpt2-small.jsonl"            # Where to save locally

# Make sure your key is set
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"

def download_dpo_file():
    print("🔄 Connecting to GCS...")
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)

    print(f"⬇️ Downloading gs://{BUCKET_NAME}/{BLOB_PATH} → {LOCAL_PATH}")
    blob = bucket.blob(BLOB_PATH)

    blob.download_to_filename(LOCAL_PATH)

    print("✅ Download complete!")

if __name__ == "__main__":
    download_dpo_file()


🔄 Connecting to GCS...
⬇️ Downloading gs://gpt2-ultrafineweb/qa-gpt2-small.jsonl → qa-gpt2-small.jsonl
✅ Download complete!


In [18]:
!torchrun --nproc_per_node=2 sft-qa.py

W1125 04:01:01.842000 290 torch/distributed/run.py:792] 
W1125 04:01:01.842000 290 torch/distributed/run.py:792] *****************************************
W1125 04:01:01.842000 290 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1125 04:01:01.842000 290 torch/distributed/run.py:792] *****************************************
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` 

In [17]:
# Always do this when changing datasets:
!rm -f qa_train_tokens.pt qa_val_tokens.pt

In [11]:
"""
Download and filter TriviaQA dataset for GPT-2 124M SFT training
Creates a clean JSONL file with short Q&A pairs
"""

import json
from datasets import load_dataset
from tqdm import tqdm

# Config
OUTPUT_FILE = "triviaqa_qa_filtered.jsonl"
MAX_ANSWER_LENGTH = 100  # Filter out long answers
MAX_QUESTION_LENGTH = 200  # Filter out long questions
TARGET_SAMPLES = 20000  # How many samples to keep (increase if you want more)

print("📥 Downloading TriviaQA dataset...")
dataset = load_dataset("trivia_qa", "unfiltered.nocontext", split="train")

print(f"📊 Total samples: {len(dataset):,}")
print(f"🔍 Filtering for short Q&A pairs...")

filtered_pairs = []

for item in tqdm(dataset):
    question = item.get("question", "").strip()
    
    # Get answer (TriviaQA has multiple answer aliases)
    answer_dict = item.get("answer", {})
    answer = answer_dict.get("value", "").strip()
    
    # Filter criteria
    if not question or not answer:
        continue
    
    if len(question) > MAX_QUESTION_LENGTH:
        continue
    
    if len(answer) > MAX_ANSWER_LENGTH:
        continue
    
    # Skip if answer has newlines (usually long explanations)
    if "\n" in answer:
        continue
    
    # Create clean Q&A pair
    qa_pair = {
        "question": question,
        "answer": answer
    }
    
    filtered_pairs.append(qa_pair)
    
    # Stop if we have enough
    if len(filtered_pairs) >= TARGET_SAMPLES:
        break

print(f"✅ Filtered samples: {len(filtered_pairs):,}")

# Save to JSONL
print(f"💾 Saving to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for pair in filtered_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + '\n')

print(f"✅ Done! Created {OUTPUT_FILE} with {len(filtered_pairs):,} Q&A pairs")
print(f"\n📊 Statistics:")
print(f"   - Questions: {len(filtered_pairs):,}")
print(f"   - Avg question length: {sum(len(p['question']) for p in filtered_pairs) / len(filtered_pairs):.1f} chars")
print(f"   - Avg answer length: {sum(len(p['answer']) for p in filtered_pairs) / len(filtered_pairs):.1f} chars")

# Show samples
print(f"\n🔮 Sample Q&A pairs:")
for i, pair in enumerate(filtered_pairs[:5]):
    print(f"\n[{i+1}] Q: {pair['question']}")
    print(f"    A: {pair['answer']}")



📥 Downloading TriviaQA dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

unfiltered.nocontext/train-00000-of-0000(…):   0%|          | 0.00/33.2M [00:00<?, ?B/s]

unfiltered.nocontext/validation-00000-of(…):   0%|          | 0.00/4.39M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/11313 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10832 [00:00<?, ? examples/s]

📊 Total samples: 87,622
🔍 Filtering for short Q&A pairs...


 23%|██▎       | 20097/87622 [00:02<00:09, 7422.09it/s]


✅ Filtered samples: 20,000
💾 Saving to triviaqa_qa_filtered.jsonl...
✅ Done! Created triviaqa_qa_filtered.jsonl with 20,000 Q&A pairs

📊 Statistics:
   - Questions: 20,000
   - Avg question length: 69.7 chars
   - Avg answer length: 11.3 chars

🔮 Sample Q&A pairs:

[1] Q: Who was President when the first Peanuts cartoon was published?
    A: Harry Truman

[2] Q: Which American-born Sinclair won the Nobel Prize for Literature in 1930?
    A: Sinclair Lewis

[3] Q: Where in England was Dame Judi Dench born?
    A: York

[4] Q: William Christensen of Madison, New Jersey, has claimed to have the world's biggest collection of what?
    A: Beer Cans

[5] Q: In which decade did Billboard magazine first publish and American hit chart?
    A: 30s


In [12]:
"""
Download and filter SQuAD dataset for GPT-2 124M SFT training
Filters for short, factual answers only
"""

import json
from datasets import load_dataset
from tqdm import tqdm

# Config
OUTPUT_FILE = "squad_qa_filtered.jsonl"
MAX_ANSWER_LENGTH = 80  # SQuAD answers can be long, filter aggressively
MAX_QUESTION_LENGTH = 150
TARGET_SAMPLES = 20000

print("📥 Downloading SQuAD v2 dataset...")
dataset = load_dataset("squad_v2", split="train")

print(f"📊 Total samples: {len(dataset):,}")
print(f"🔍 Filtering for short Q&A pairs...")

filtered_pairs = []

for item in tqdm(dataset):
    question = item.get("question", "").strip()
    
    # SQuAD format has answers as list of dicts
    answers = item.get("answers", {})
    answer_list = answers.get("text", [])
    
    # Skip if no answer (SQuAD v2 has unanswerable questions)
    if not answer_list:
        continue
    
    # Take first answer
    answer = answer_list[0].strip()
    
    # Filter criteria
    if not question or not answer:
        continue
    
    if len(question) > MAX_QUESTION_LENGTH:
        continue
    
    if len(answer) > MAX_ANSWER_LENGTH:
        continue
    
    # Skip if answer is multiple sentences (too complex)
    if answer.count('.') > 1:
        continue
    
    # Skip if answer has question marks (usually not simple factual)
    if '?' in answer:
        continue
    
    # Create clean Q&A pair
    qa_pair = {
        "question": question,
        "answer": answer
    }
    
    filtered_pairs.append(qa_pair)
    
    # Stop if we have enough
    if len(filtered_pairs) >= TARGET_SAMPLES:
        break

print(f"✅ Filtered samples: {len(filtered_pairs):,}")

# Save to JSONL
print(f"💾 Saving to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for pair in filtered_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + '\n')

print(f"✅ Done! Created {OUTPUT_FILE} with {len(filtered_pairs):,} Q&A pairs")
print(f"\n📊 Statistics:")
print(f"   - Questions: {len(filtered_pairs):,}")
print(f"   - Avg question length: {sum(len(p['question']) for p in filtered_pairs) / len(filtered_pairs):.1f} chars")
print(f"   - Avg answer length: {sum(len(p['answer']) for p in filtered_pairs) / len(filtered_pairs):.1f} chars")

# Show samples
print(f"\n🔮 Sample Q&A pairs:")
for i, pair in enumerate(filtered_pairs[:5]):
    print(f"\n[{i+1}] Q: {pair['question']}")
    print(f"    A: {pair['answer']}")



📥 Downloading SQuAD v2 dataset...


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

📊 Total samples: 130,319
🔍 Filtering for short Q&A pairs...


 21%|██        | 27117/130319 [00:01<00:06, 15273.38it/s]


✅ Filtered samples: 20,000
💾 Saving to squad_qa_filtered.jsonl...
✅ Done! Created squad_qa_filtered.jsonl with 20,000 Q&A pairs

📊 Statistics:
   - Questions: 20,000
   - Avg question length: 59.8 chars
   - Avg answer length: 14.8 chars

🔮 Sample Q&A pairs:

[1] Q: When did Beyonce start becoming popular?
    A: in the late 1990s

[2] Q: What areas did Beyonce compete in when she was growing up?
    A: singing and dancing

[3] Q: When did Beyonce leave Destiny's Child and become a solo singer?
    A: 2003

[4] Q: In what city and state did Beyonce  grow up?
    A: Houston, Texas

[5] Q: In which decade did Beyonce become famous?
    A: late 1990s


In [13]:
"""
Merge multiple Q&A datasets into one large training file
This combines your custom dataset with downloaded datasets
"""

import json
import random
from pathlib import Path

# Config
SEED = 1337
OUTPUT_FILE = "combined_qa_dataset.jsonl"

# Input files (comment out ones you don't have)
INPUT_FILES = [
    "qa-gpt2-small.jsonl",           # Your original (1,336 samples)
    "triviaqa_qa_filtered.jsonl",  # TriviaQA (20k samples) - uncomment after download
    "squad_qa_filtered.jsonl",     # SQuAD (20k samples) - uncomment after download
]

print("📚 Merging Q&A datasets...")
print("="*60)

all_pairs = []
stats = {}

for input_file in INPUT_FILES:
    if not Path(input_file).exists():
        print(f"⚠️  Skipping {input_file} (not found)")
        continue
    
    count = 0
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                pair = json.loads(line)
                all_pairs.append(pair)
                count += 1
    
    stats[input_file] = count
    print(f"✅ Loaded {count:,} pairs from {input_file}")

print("="*60)
print(f"📊 Total pairs: {len(all_pairs):,}")
print()

# Shuffle with seed
random.seed(SEED)
random.shuffle(all_pairs)

# Remove duplicates based on question
print("🔍 Removing duplicate questions...")
seen_questions = set()
unique_pairs = []

for pair in all_pairs:
    question = pair.get("question", "").strip().lower()
    if question not in seen_questions:
        seen_questions.add(question)
        unique_pairs.append(pair)

print(f"✅ Unique pairs: {len(unique_pairs):,}")
print(f"   (removed {len(all_pairs) - len(unique_pairs):,} duplicates)")
print()

# Save merged dataset
print(f"💾 Saving to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for pair in unique_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + '\n')

print(f"✅ Done! Created {OUTPUT_FILE}")
print()
print("="*60)
print("📊 FINAL STATISTICS")
print("="*60)
for filename, count in stats.items():
    print(f"  {filename}: {count:,} pairs")
print(f"\n  Total unique: {len(unique_pairs):,} pairs")
print("="*60)
print()
print(f"🎯 Now use: {OUTPUT_FILE} for training!")
print(f"   Edit sft_qa_ddp.py and set:")
print(f"   QA_JSONL_FILE = '{OUTPUT_FILE}'")



📚 Merging Q&A datasets...
✅ Loaded 1,336 pairs from qa-gpt2-small.jsonl
✅ Loaded 20,000 pairs from triviaqa_qa_filtered.jsonl
✅ Loaded 20,000 pairs from squad_qa_filtered.jsonl
📊 Total pairs: 41,336

🔍 Removing duplicate questions...
✅ Unique pairs: 41,238
   (removed 98 duplicates)

💾 Saving to combined_qa_dataset.jsonl...
✅ Done! Created combined_qa_dataset.jsonl

📊 FINAL STATISTICS
  qa-gpt2-small.jsonl: 1,336 pairs
  triviaqa_qa_filtered.jsonl: 20,000 pairs
  squad_qa_filtered.jsonl: 20,000 pairs

  Total unique: 41,238 pairs

🎯 Now use: combined_qa_dataset.jsonl for training!
   Edit sft_qa_ddp.py and set:
   QA_JSONL_FILE = 'combined_qa_dataset.jsonl'


In [31]:
"""
SFT Training for GPT-2 124M on CLEAN 1.3k Q&A Dataset
Optimized for small, high-quality dataset
"""

import os
import sys
import math
import time
import json
import tiktoken
from tqdm import tqdm
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# Import your GPT model
sys.path.append('/home/shubharthak/Desktop/karpathy/chapter-8-building-gpt2-124m')
from train_gpt2_v2 import GPT, GPTConfig

# WandB & GCS
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

try:
    from google.cloud import storage
    GCS_AVAILABLE = True
except ImportError:
    GCS_AVAILABLE = False

# ============================================================
# CONFIG - OPTIMIZED FOR 1.3K CLEAN DATASET
# ============================================================

QA_JSONL_FILE = "qa-gpt2-small.jsonl"  # 1,336 clean samples
PRETRAINED_CHECKPOINT = "checkpoints/model_09535.pt"

# Training hyperparameters - optimized for small dataset
BATCH_SIZE = 8              # Smaller batch for 1.3k dataset
SEQ_LEN = 256
GRAD_ACCUM_STEPS = 16       # Effective batch = 8 * 16 = 128
LR = 5e-6                   # Lower LR for careful learning
MIN_LR = 5e-8
WEIGHT_DECAY = 0.1
MAX_STEPS = 800             # Fewer steps for smaller dataset
WARMUP_STEPS = 80           # 10% warmup

# Evaluation & Checkpointing
EVAL_EVERY = 40
SAVE_EVERY = 80
PATIENCE = 6                # Stop after 6 evals without improvement
EARLY_STOP_ENABLED = True

# WandB
WANDB_PROJECT = "gpt2-qa-sft"
WANDB_RUN_NAME = "qa-sft-gpt2-124m-clean-1.3k"

# GCS
GCS_BUCKET = "gpt2-ultrafineweb"
GCS_PREFIX = "qa_sft_checkpoints"
GCS_KEY_PATH = "gpt2-bucket-key.json"

# Output
CHECKPOINT_DIR = "qa_sft_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ============================================================
# GCS Upload
# ============================================================

def upload_to_gcs(local_path: str, gcs_path: str):
    """Upload file to Google Cloud Storage"""
    if not GCS_AVAILABLE:
        return
    
    try:
        if os.path.exists(GCS_KEY_PATH):
            os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = GCS_KEY_PATH
            client = storage.Client()
            bucket = client.bucket(GCS_BUCKET)
            blob = bucket.blob(gcs_path)
            blob.upload_from_filename(local_path)
            print(f"☁️ Uploaded: gs://{GCS_BUCKET}/{gcs_path}")
    except Exception as e:
        print(f"⚠️  GCS upload failed: {e}")

# ============================================================
# Dataset Loading & Tokenization
# ============================================================

def load_qa_dataset(jsonl_file: str, train_split: float = 0.85):
    """Load Q&A pairs from JSONL file"""
    print(f"\n📖 Loading Q&A dataset from {jsonl_file}...")
    
    qa_pairs = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                qa_pairs.append({
                    'question': data['question'],
                    'answer': data['answer']
                })
    
    print(f"📊 Total Q&A pairs: {len(qa_pairs)}")
    
    # Shuffle and split
    import random
    random.seed(42)
    random.shuffle(qa_pairs)
    
    split_idx = int(len(qa_pairs) * train_split)
    train_data = qa_pairs[:split_idx]
    val_data = qa_pairs[split_idx:]
    
    print(f"📊 Train samples: {len(train_data)} ({train_split*100:.0f}%)")
    print(f"📊 Val samples: {len(val_data)} ({(1-train_split)*100:.0f}%)")
    
    return train_data, val_data

def tokenize_qa_dataset(qa_pairs: List[Dict], enc: tiktoken.Encoding):
    """Tokenize Q&A pairs with simple format"""
    all_tokens = []
    
    for qa in tqdm(qa_pairs, desc="Tokenizing"):
        # Simple format: Q: <question>\nA: <answer>
        text = f"Q: {qa['question']}\nA: {qa['answer']}"
        
        # Tokenize
        tokens = enc.encode_ordinary(text)
        tokens.append(enc.eot_token)  # End of text
        all_tokens.extend(tokens)
    
    return torch.tensor(all_tokens, dtype=torch.long)

# ============================================================
# DataLoader
# ============================================================

class QADataLoader:
    def __init__(self, tokens: torch.Tensor, B: int, T: int, process_rank: int, num_processes: int):
        self.tokens = tokens
        self.B = B
        self.T = T
        self.process_rank = process_rank
        self.num_processes = num_processes
        
        # Calculate tokens per process
        self.num_batches = len(tokens) // (B * T * num_processes)
        self.current_position = process_rank * B * T
    
    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        
        # Check if we have enough tokens
        if len(buf) < B * T + 1:
            self.current_position = self.process_rank * B * T
            buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        
        self.current_position += B * T * self.num_processes
        
        if self.current_position + B * T * self.num_processes + 1 > len(self.tokens):
            self.current_position = self.process_rank * B * T
        
        return x, y
    
    def reset(self):
        self.current_position = self.process_rank * self.B * self.T

# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def eval_loss(model, val_loader, device, max_steps=20):
    """Evaluate validation loss"""
    model.eval()
    val_loader.reset()
    
    total_loss = 0.0
    for _ in range(max_steps):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model(x, y)
        
        total_loss += loss.item()
    
    avg_loss = total_loss / max_steps
    model.train()
    return avg_loss

# ============================================================
# Sample Generation
# ============================================================

@torch.no_grad()
def generate_samples(model, enc, device, prompts, max_length=100, temperature=0.8, top_k=30):
    """Generate sample completions"""
    model.eval()
    samples = []
    
    for prompt in prompts:
        tokens = enc.encode_ordinary(prompt)
        tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
        
        # Generate
        for _ in range(max_length):
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits, _ = model(tokens)
            
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)
            
            # Stop at EOS
            if next_token.item() == enc.eot_token:
                break
        
        generated = enc.decode(tokens[0].tolist())
        samples.append(generated)
    
    model.train()
    return samples

# ============================================================
# Main Training Loop
# ============================================================

def train_qa_sft_ddp():
    # DDP setup
    ddp = int(os.environ.get('RANK', -1)) != -1
    if ddp:
        assert torch.cuda.is_available()
        dist.init_process_group(backend='nccl')
        ddp_rank = int(os.environ['RANK'])
        ddp_local_rank = int(os.environ['LOCAL_RANK'])
        ddp_world_size = int(os.environ['WORLD_SIZE'])
        device = f'cuda:{ddp_local_rank}'
        torch.cuda.set_device(device)
        master_process = (ddp_rank == 0)
    else:
        ddp_rank = 0
        ddp_local_rank = 0
        ddp_world_size = 1
        master_process = True
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)
    
    # WandB init (master only)
    if master_process and WANDB_AVAILABLE:
        wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, config={
            'dataset': QA_JSONL_FILE,
            'batch_size': BATCH_SIZE,
            'seq_len': SEQ_LEN,
            'grad_accum': GRAD_ACCUM_STEPS,
            'lr': LR,
            'max_steps': MAX_STEPS,
        })
    
    # Load dataset
    if master_process:
        train_data, val_data = load_qa_dataset(QA_JSONL_FILE)
        enc = tiktoken.get_encoding("gpt2")
        
        # Tokenize
        train_tokens = tokenize_qa_dataset(train_data, enc)
        val_tokens = tokenize_qa_dataset(val_data, enc)
        
        print(f"✅ Train tokens: {len(train_tokens):,}")
        print(f"✅ Val tokens: {len(val_tokens):,}")
        
        # Save tokens
        torch.save(train_tokens, "qa_train_tokens.pt")
        torch.save(val_tokens, "qa_val_tokens.pt")
    
    if ddp:
        dist.barrier()
    
    # Load tokens
    enc = tiktoken.get_encoding("gpt2")
    train_tokens = torch.load("qa_train_tokens.pt", weights_only=True)
    val_tokens = torch.load("qa_val_tokens.pt", weights_only=True)
    
    # Create dataloaders
    train_loader = QADataLoader(train_tokens, BATCH_SIZE, SEQ_LEN, ddp_rank, ddp_world_size)
    val_loader = QADataLoader(val_tokens, BATCH_SIZE, SEQ_LEN, ddp_rank, ddp_world_size)
    
    # Load model
    if master_process:
        print(f"\n🔄 Loading checkpoint: {PRETRAINED_CHECKPOINT}\n")
    
    checkpoint = torch.load(PRETRAINED_CHECKPOINT, map_location=device, weights_only=False)
    model = GPT(GPTConfig(vocab_size=50304))
    model.load_state_dict(checkpoint['model'])
    model = model.to(device)
    model.train()
    
    if ddp:
        model = DDP(model, device_ids=[ddp_local_rank])
    raw_model = model.module if ddp else model
    
    if master_process:
        print(f"✅ Model loaded. start_step=0")
    
    # Optimizer
    optimizer = raw_model.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=LR,
        device_type='cuda'
    )
    
    # Training state
    best_val_loss = float('inf')
    patience_counter = 0
    
    # Print config
    if master_process:
        print("\n" + "="*60)
        print("🚀 Starting Q&A SFT Training - CLEAN 1.3K DATASET")
        print("="*60)
        print(f"📊 Dataset: {QA_JSONL_FILE}")
        print(f"📊 Total samples: 1,336 Q&A pairs")
        print(f"📊 Train tokens: {len(train_tokens):,}")
        print(f"📊 Val tokens: {len(val_tokens):,}")
        print(f"📊 Batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
        print(f"📊 Sequence length: {SEQ_LEN}")
        print(f"📊 Learning rate: {LR:.2e}")
        print(f"📊 Max steps: {MAX_STEPS}")
        print(f"📊 Eval every: {EVAL_EVERY} steps")
        print(f"📊 Save every: {SAVE_EVERY} steps")
        print(f"📊 Early stop patience: {PATIENCE} evals")
        print("="*60 + "\n")
    
    # Training loop
    for step in range(MAX_STEPS):
        t0 = time.time()
        last_step = (step == MAX_STEPS - 1)
        
        # Evaluation
        if (step % EVAL_EVERY == 0 and step > 0) or last_step:
            val_loss = eval_loss(raw_model, val_loader, device)
            
            if ddp:
                dist.all_reduce(torch.tensor([val_loss], device=device), op=dist.ReduceOp.AVG)
            
            if master_process:
                ppl = math.exp(val_loss)
                print(f"\n{'='*60}")
                print(f"📊 EVALUATION @ step {step}")
                print(f"   Val loss: {val_loss:.6f} | Perplexity: {ppl:.2f}")
                print("="*60)
                
                # Check for improvement
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    print(f"💾 ✨ NEW BEST MODEL! val_loss: {val_loss:.4f}")
                    
                    # Save best
                    best_path = os.path.join(CHECKPOINT_DIR, "qa-sft_best.pt")
                    torch.save({
                        'model': raw_model.state_dict(),
                        'step': step,
                        'val_loss': val_loss,
                    }, best_path)
                    upload_to_gcs(best_path, f"{GCS_PREFIX}/qa-sft_best.pt")
                else:
                    patience_counter += 1
                    print(f"⚠️  Val loss did not improve. Patience: {patience_counter}/{PATIENCE}")
                    print(f"   (best: {best_val_loss:.4f} @ step {step - patience_counter * EVAL_EVERY})")
                
                # Early stopping
                if EARLY_STOP_ENABLED and patience_counter >= PATIENCE:
                    print(f"\n🛑 EARLY STOPPING triggered after {patience_counter} evaluations without improvement")
                    print(f"🎯 Best val loss: {best_val_loss:.4f}")
                    print(f"✅ Best model saved at: {CHECKPOINT_DIR}/qa-sft_best.pt\n")
                    break
                
                # Sample generations
                print(f"\n{'='*60}")
                print("🔮 SAMPLE GENERATIONS")
                print("="*60 + "\n")
                
                test_prompts = [
                    "Q: What is the capital of Australia?\nA:",
                    "Q: What is RAM?\nA:",
                    "Q: What does 'arvo' mean?\nA:",
                    "Q: What is a koala's main food?\nA:",
                    "Q: What is overfitting?\nA:",
                ]
                
                samples = generate_samples(raw_model, enc, device, test_prompts, max_length=80, temperature=0.8, top_k=30)
                for i, sample in enumerate(samples, 1):
                    print(f"[{i}] {sample}\n")
                print("="*60 + "\n")
                
                # WandB logging
                if WANDB_AVAILABLE:
                    wandb.log({
                        'val_loss': val_loss,
                        'val_perplexity': ppl,
                        'step': step,
                        'patience': patience_counter,
                    })
        
        # Save checkpoint
        if master_process and step > 0 and step % SAVE_EVERY == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"qa-sft_{step:05d}.pt")
            torch.save({
                'model': raw_model.state_dict(),
                'step': step,
            }, ckpt_path)
            print(f"💾 Saved checkpoint: {ckpt_path}")
            upload_to_gcs(ckpt_path, f"{GCS_PREFIX}/qa-sft_{step:05d}.pt")
        
        # Training step
        model.train()
        optimizer.zero_grad()
        loss_accum = 0.0
        
        for micro_step in range(GRAD_ACCUM_STEPS):
            x, y = train_loader.next_batch()
            x, y = x.to(device), y.to(device)
            
            # Automatic attention kernel selection (fallback for older GPUs)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits, loss = model(x, y)
            
            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.detach()
            
            if ddp:
                model.require_backward_grad_sync = (micro_step == GRAD_ACCUM_STEPS - 1)
            
            loss.backward()
        
        if ddp:
            dist.all_reduce(loss_accum, op=dist.ReduceOp.AVG)
        
        # Gradient clipping
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Learning rate schedule (cosine with warmup)
        if step < WARMUP_STEPS:
            lr = LR * (step + 1) / WARMUP_STEPS
        else:
            decay_ratio = (step - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
            coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
            lr = MIN_LR + coeff * (LR - MIN_LR)
        
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        
        optimizer.step()
        
        torch.cuda.synchronize()
        t1 = time.time()
        dt = (t1 - t0) * 1000
        tokens_per_sec = (BATCH_SIZE * SEQ_LEN * GRAD_ACCUM_STEPS * ddp_world_size) / (t1 - t0)
        
        if master_process:
            print(f"step {step:4d} | loss {loss_accum.item():.4f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt:.0f}ms | tok/s {tokens_per_sec:.0f}")
            
            if WANDB_AVAILABLE:
                wandb.log({
                    'train_loss': loss_accum.item(),
                    'learning_rate': lr,
                    'grad_norm': norm,
                    'step': step,
                })
    
    if ddp:
        dist.destroy_process_group()
    
    if master_process and WANDB_AVAILABLE:
        wandb.finish()

if __name__ == "__main__":
    train_qa_sft_ddp()



/usr/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value Tr

Service Account Key Loaded
Auth OK, bucket exists: <Bucket: gpt2-ultrafineweb>



📖 Loading Q&A dataset from qa-gpt2-small.jsonl...
📊 Total Q&A pairs: 1336
📊 Train samples: 1135 (85%)
📊 Val samples: 201 (15%)


Tokenizing: 100%|██████████| 201/201 [00:00<00:00, 71384.85it/s]


✅ Train tokens: 23,034
✅ Val tokens: 4,126

🔄 Loading checkpoint: checkpoints/model_09535.pt

✅ Model loaded. start_step=0
num decayed parameter tensors: 50, with 124,354,560 parameters
num non-decayed parameter tensors: 98, with 121,344 parameters
using fused AdamW: True

🚀 Starting Q&A SFT Training - CLEAN 1.3K DATASET
📊 Dataset: qa-gpt2-small.jsonl
📊 Total samples: 1,336 Q&A pairs
📊 Train tokens: 23,034
📊 Val tokens: 4,126
📊 Batch size: 8 x 16 = 128
📊 Sequence length: 256
📊 Learning rate: 5.00e-06
📊 Max steps: 800
📊 Eval every: 40 steps
📊 Save every: 80 steps
📊 Early stop patience: 6 evals

step    0 | loss 3.0995 | lr 6.25e-08 | norm 8.67 | dt 10001ms | tok/s 3276
step    1 | loss 3.0862 | lr 1.25e-07 | norm 8.70 | dt 9843ms | tok/s 3329
step    2 | loss 3.0962 | lr 1.88e-07 | norm 8.66 | dt 9991ms | tok/s 3280
step    3 | loss 3.0748 | lr 2.50e-07 | norm 8.62 | dt 10084ms | tok/s 3249
step    4 | loss 3.0836 | lr 3.13e-07 | norm 8.63 | dt 10234ms | tok/s 3202
step    5 | loss 3.06

KeyboardInterrupt: 

In [ ]:
!bash train_clean.sh

In [20]:
%%writefile train_clean.sh
#!/bin/bash

# SFT Training on Clean 1.3k Q&A Dataset
# Optimized for GPT-2 124M with small, high-quality data

echo "============================================================"
echo "🚀 SFT Training - Clean 1.3k Q&A Dataset"
echo "============================================================"
echo ""

# Delete old token files to force regeneration
if [ -f "qa_train_tokens.pt" ] || [ -f "qa_val_tokens.pt" ]; then
    echo "🧹 Cleaning up old token files..."
    rm -f qa_train_tokens.pt qa_val_tokens.pt
    echo "✅ Old tokens removed"
    echo ""
fi

# Check dataset exists
if [ ! -f "qa-gpt2-small.jsonl" ]; then
    echo "❌ Error: qa-gpt2-small.jsonl not found!"
    exit 1
fi

# Count Q&A pairs
NUM_SAMPLES=$(wc -l < qa-gpt2-small.jsonl)
echo "📊 Dataset: qa-gpt2-small.jsonl ($NUM_SAMPLES samples)"
echo ""

# Detect number of GPUs
NUM_GPUS=$(nvidia-smi --list-gpus 2>/dev/null | wc -l)

if [ $NUM_GPUS -eq 0 ]; then
    echo "⚠️  No GPUs detected. Running on CPU (slow!)"
    python sft-qa-clean-1k.py
elif [ $NUM_GPUS -eq 1 ]; then
    echo "🔧 Single GPU detected"
    echo "▶️  Starting training..."
    echo ""
    python sft-qa-clean-1k.py
else
    echo "🔧 $NUM_GPUS GPUs detected"
    echo "▶️  Starting DDP training..."
    echo ""
    torchrun --standalone --nproc_per_node=$NUM_GPUS sft-qa-clean-1k.py
fi



Writing train_clean.sh


In [14]:
%%writefile sft-qa.py
import os, math, random, time, json
from tqdm import tqdm

import torch
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

import tiktoken
import wandb
from google.cloud import storage

# Flash Attention v1 ON (T4 supports v1)
from torch.backends.cuda import sdp_kernel
sdp_kernel(enable_flash=True, enable_mem_efficient=False, enable_math=False)

# -----------------------------
# Config
# -----------------------------
QA_JSONL_FILE = "combined_qa_dataset.jsonl"  # ✅ USING COMBINED DATASET (41k samples)

OUT_DIR = "qa_sft_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_CKPT   = "checkpoints/model_09535.pt"      # pretrained checkpoint
RESUME_CKPT = None  # e.g. "qa_sft_checkpoints/qa-sft_00400.pt" if resuming

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ✅ OPTIMIZED Hyperparams for GPT-2 124M with LARGE DATASET (41k samples)
BATCH_SIZE = 16          # ✅ Increased from 8 (we have more data now!)
SEQ_LEN = 256            # ✅ Short sequences (Q&A are brief)
GRAD_ACCUM_STEPS = 8     # ✅ Total batch = 16*8 = 128 (same effective batch size)
LR = 1e-5                # ✅ Keep conservative LR
MIN_LR = 1e-7            # ✅ Low minimum
WEIGHT_DECAY = 0.1       # ✅ Regularization
MAX_STEPS = 2500         # ✅ INCREASED from 1500 (we have 30x more data!)
WARMUP_STEPS = 150       # ✅ INCREASED from 100 (longer warmup for more data)

EVAL_EVERY = 50          # ✅ Evaluate every 50 steps (slightly less frequent)
SAVE_EVERY = 100         # ✅ Save every 100 steps (less frequent saves)
SEED = 1337

# ✅ EARLY STOPPING
PATIENCE = 8             # ✅ INCREASED from 5 (more patience with larger dataset)
EARLY_STOP_ENABLED = True

# Train/Val Split
TRAIN_VAL_SPLIT = 0.85   # 85% train, 15% val

# W&B
WANDB_PROJECT = "gpt2-qa-sft"
WANDB_RUN_NAME = "qa-sft-gpt2-124m-combined-41k"  # ✅ Updated run name

# GCS
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET_NAME = "gpt2-ultrafineweb"
GCS_PREFIX = "qa_sft_checkpoints"

def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET_NAME)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET_NAME}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# -----------------------------
# Import your model
# -----------------------------
from train_gpt2_v2 import GPT, GPTConfig

enc = tiktoken.get_encoding("gpt2")
EOS_ID = enc.eot_token

# -----------------------------
# Load and Split Q&A Dataset
# -----------------------------
def load_qa_dataset(master_process: bool):
    """Load Q&A JSONL file and split into train/val"""
    
    if os.path.exists("qa_train_tokens.pt") and os.path.exists("qa_val_tokens.pt"):
        if master_process:
            print("✅ Token files already exist.")
        if dist.is_initialized():
            dist.barrier()
        
        # Load and print stats
        train_tokens = torch.load("qa_train_tokens.pt")
        val_tokens = torch.load("qa_val_tokens.pt")
        if master_process:
            print(f"📊 Train tokens: {len(train_tokens):,}")
            print(f"📊 Val tokens: {len(val_tokens):,}")
        return

    if master_process:
        print(f"\n📖 Loading Q&A dataset from {QA_JSONL_FILE}...")
        
        # Load all Q&A pairs
        qa_pairs = []
        with open(QA_JSONL_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    qa_pairs.append(json.loads(line))
        
        total_samples = len(qa_pairs)
        print(f"📊 Total Q&A pairs: {total_samples}")
        
        # Shuffle with seed
        random.seed(SEED)
        random.shuffle(qa_pairs)
        
        # Split train/val
        split_idx = int(total_samples * TRAIN_VAL_SPLIT)
        train_pairs = qa_pairs[:split_idx]
        val_pairs = qa_pairs[split_idx:]
        
        print(f"📊 Train samples: {len(train_pairs)} ({TRAIN_VAL_SPLIT*100:.0f}%)")
        print(f"📊 Val samples: {len(val_pairs)} ({(1-TRAIN_VAL_SPLIT)*100:.0f}%)")
        
        # Tokenize function - SIMPLE FORMAT (no Alpaca!)
        def tokenize_qa_pairs(pairs, desc):
            all_ids = []
            for pair in tqdm(pairs, desc=desc):
                question = pair.get("question", "").strip()
                answer = pair.get("answer", "").strip()
                
                # ✅ SIMPLE FORMAT: Q: ... A: ...
                text = f"Q: {question}\nA: {answer}"
                
                ids = enc.encode(text)
                ids.append(EOS_ID)  # Add EOS token
                all_ids.extend(ids)
            
            return torch.tensor(all_ids, dtype=torch.long)
        
        # Tokenize train and val
        train_tokens = tokenize_qa_pairs(train_pairs, "Tokenizing train")
        val_tokens = tokenize_qa_pairs(val_pairs, "Tokenizing val")
        
        # Save
        torch.save(train_tokens, "qa_train_tokens.pt")
        torch.save(val_tokens, "qa_val_tokens.pt")
        
        print(f"✅ Train tokens: {len(train_tokens):,}")
        print(f"✅ Val tokens: {len(val_tokens):,}")
        print("✅ Saved qa_train_tokens.pt and qa_val_tokens.pt")

    if dist.is_initialized():
        dist.barrier()

# -----------------------------
# DDP TokenLoader
# -----------------------------
class TokenLoaderDDP:
    def __init__(self, tokens, batch_size, seq_len, process_rank, num_processes):
        self.tokens = tokens
        self.B = batch_size
        self.T = seq_len
        self.rank = process_rank
        self.world = num_processes
        self.pos = self.B * self.T * self.rank

    def next_batch(self):
        B, T = self.B, self.T
        need = B*T + 1
        if self.pos + need >= len(self.tokens):
            self.pos = self.B * self.T * self.rank
        buf = self.tokens[self.pos : self.pos + need]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        self.pos += B*T*self.world
        return x, y

# -----------------------------
# Load model (pretrain or resume)
# -----------------------------
def load_model(device):
    ckpt_path = RESUME_CKPT if (RESUME_CKPT and os.path.exists(RESUME_CKPT)) else BASE_CKPT
    print(f"\n🔄 Loading checkpoint: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config: GPTConfig = ckpt["config"]

    model = GPT(config)
    model.load_state_dict(ckpt["model"])
    model.to(device).train()

    start_step = int(ckpt.get("step", 0))

    # Reset step counter if loading from base pretrained checkpoint
    if ckpt_path == BASE_CKPT:
        start_step = 0

    print(f"✅ Model loaded. start_step={start_step}")
    return model, config, start_step

# -----------------------------
# Cosine LR schedule
# -----------------------------
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, (MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + cosine * (LR - MIN_LR)

# -----------------------------
# Eval + sample
# -----------------------------
@torch.no_grad()
def eval_loss(model, val_loader, device, batches=30):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with amp.autocast("cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses)

@torch.no_grad()
def sample_model(model, prompt, device, max_new=80, top_k=50):
    """Generate sample from model"""
    model.eval()
    ids = enc.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long)[None, :].to(device)

    for _ in range(max_new):
        logits, _ = model(x)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, k=top_k, dim=-1)
        ix = torch.multinomial(topk_probs, 1)
        next_id = torch.gather(topk_idx, -1, ix)
        x = torch.cat([x, next_id], dim=1)
        if next_id.item() == EOS_ID:
            break

    text = enc.decode(x[0].tolist())
    model.train()
    return text

# -----------------------------
# Main train
# -----------------------------
def train_qa_sft_ddp():
    ddp = int(os.environ.get("RANK", -1)) != -1
    if ddp:
        dist.init_process_group("nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
        master = (rank == 0)
    else:
        rank, local_rank, world_size = 0, 0, 1
        device = DEVICE
        master = True

    random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    torch.cuda.manual_seed(SEED + rank)

    # Load and prepare dataset
    load_qa_dataset(master)

    train_tokens = torch.load("qa_train_tokens.pt")
    val_tokens   = torch.load("qa_val_tokens.pt")

    train_loader = TokenLoaderDDP(train_tokens, BATCH_SIZE, SEQ_LEN, rank, world_size)
    val_loader   = TokenLoaderDDP(val_tokens,   BATCH_SIZE, SEQ_LEN, rank, world_size)

    model, config, start_step = load_model(device)
    if ddp:
        model = DDP(model, device_ids=[local_rank])
    raw_model = model.module if ddp else model

    optimizer = raw_model.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=LR,
        device_type=device
    )
    scaler = amp.GradScaler(device="cuda")

    # ✅ Early stopping variables
    best_val_loss = float('inf')
    patience_counter = 0
    best_step = 0

    if master:
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=dict(
                dataset=QA_JSONL_FILE,
                dataset_size="41,238 samples",
                base_ckpt=BASE_CKPT,
                resume_ckpt=RESUME_CKPT,
                batch_size=BATCH_SIZE,
                seq_len=SEQ_LEN,
                grad_accum_steps=GRAD_ACCUM_STEPS,
                total_batch_size=BATCH_SIZE * GRAD_ACCUM_STEPS,
                lr=LR,
                min_lr=MIN_LR,
                weight_decay=WEIGHT_DECAY,
                max_steps=MAX_STEPS,
                warmup_steps=WARMUP_STEPS,
                patience=PATIENCE,
                early_stop_enabled=EARLY_STOP_ENABLED,
                eval_every=EVAL_EVERY,
                save_every=SAVE_EVERY,
                train_val_split=TRAIN_VAL_SPLIT,
                ddp_world_size=world_size,
                flash_attention=True,
            )
        )

    print(f"\n{'='*60}")
    print("🚀 Starting Q&A SFT Training - LARGE DATASET")
    print(f"{'='*60}")
    print(f"📊 Dataset: {QA_JSONL_FILE}")
    print(f"📊 Total samples: 41,238 Q&A pairs")
    print(f"📊 Train tokens: {len(train_tokens):,}")
    print(f"📊 Val tokens: {len(val_tokens):,}")
    print(f"📊 Batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
    print(f"📊 Sequence length: {SEQ_LEN}")
    print(f"📊 Learning rate: {LR:.2e}")
    print(f"📊 Max steps: {MAX_STEPS}")
    print(f"📊 Eval every: {EVAL_EVERY} steps")
    print(f"📊 Save every: {SAVE_EVERY} steps")
    print(f"📊 Early stop patience: {PATIENCE} evals")
    print(f"{'='*60}\n")

    for step in range(start_step, MAX_STEPS):
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0

        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        for micro in range(GRAD_ACCUM_STEPS):
            x, y = train_loader.next_batch()
            x, y = x.to(device), y.to(device)

            with amp.autocast("cuda"):
                _, loss = model(x, y)

            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.item()

            if ddp:
                model.require_backward_grad_sync = (micro == GRAD_ACCUM_STEPS - 1)

            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()

        if ddp:
            loss_t = torch.tensor(loss_accum, device=device)
            dist.all_reduce(loss_t, op=dist.ReduceOp.AVG)
            loss_accum = loss_t.item()

        dt = time.time() - t0
        tokens_per_sec = (BATCH_SIZE * SEQ_LEN * GRAD_ACCUM_STEPS * world_size) / dt
        
        if master and step % 10 == 0:
            print(f"step {step:4d} | loss {loss_accum:.4f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt*1000:.0f}ms | tok/s {tokens_per_sec:.0f}")
            wandb.log({
                "train/loss": loss_accum, 
                "train/lr": lr, 
                "train/grad_norm": norm,
                "train/tokens_per_sec": tokens_per_sec,
                "step": step
            })

        # ✅ Evaluation every EVAL_EVERY steps
        if master and step > 0 and step % EVAL_EVERY == 0:
            vloss = eval_loss(raw_model, val_loader, device)
            ppl = math.exp(vloss) if vloss < 10 else float('inf')
            print(f"\n{'='*60}")
            print(f"📊 EVALUATION @ step {step}")
            print(f"   Val loss: {vloss:.4f} | Perplexity: {ppl:.2f}")
            print(f"{'='*60}")
            wandb.log({"val/loss": vloss, "val/perplexity": ppl, "step": step})

            # ✅ Early stopping logic
            if vloss < best_val_loss:
                best_val_loss = vloss
                patience_counter = 0
                best_step = step
                
                # Save best model with "qa-sft" prefix
                best_path = os.path.join(OUT_DIR, "qa-sft_best.pt")
                torch.save({
                    "model": raw_model.state_dict(), 
                    "config": config, 
                    "step": step,
                    "val_loss": vloss
                }, best_path)
                print(f"💾 ✨ NEW BEST MODEL! val_loss: {vloss:.4f}")
                upload_to_gcs(best_path, f"{GCS_PREFIX}/qa-sft_best.pt")
                wandb.log({"best_val_loss": vloss, "best_step": step})
            else:
                patience_counter += 1
                print(f"⚠️  Val loss did not improve. Patience: {patience_counter}/{PATIENCE}")
                print(f"   (best: {best_val_loss:.4f} @ step {best_step})")
                wandb.log({"patience_counter": patience_counter, "step": step})
                
            if EARLY_STOP_ENABLED and patience_counter >= PATIENCE:
                print(f"\n{'='*60}")
                print(f"🛑 EARLY STOPPING at step {step}")
                print(f"   Best val_loss: {best_val_loss:.4f} at step {best_step}")
                print(f"{'='*60}\n")
                break

            # ✅ Sample generation
            test_prompts = [
                "Q: What is the capital of Australia?\nA:",
                "Q: What is RAM?\nA:",
                "Q: What does 'arvo' mean?\nA:",
                "Q: What is a koala's main food?\nA:",
                "Q: What is overfitting?\nA:",
            ]
            
            print(f"\n{'='*60}")
            print("🔮 SAMPLE GENERATIONS")
            print(f"{'='*60}")
            
            samples_dict = {}
            for i, prompt in enumerate(test_prompts):
                sample = sample_model(raw_model, prompt, device)
                print(f"\n[{i+1}] {sample}")
                samples_dict[f"sample_{i+1}"] = sample
            
            print(f"{'='*60}\n")
            wandb.log(samples_dict)

        # ✅ Save checkpoint every SAVE_EVERY steps with "qa-sft" prefix
        if master and step > 0 and step % SAVE_EVERY == 0:
            ckpt_path = os.path.join(OUT_DIR, f"qa-sft_{step:05d}.pt")
            torch.save({
                "model": raw_model.state_dict(), 
                "config": config, 
                "step": step
            }, ckpt_path)
            print(f"💾 Saved checkpoint: {ckpt_path}")
            upload_to_gcs(ckpt_path, f"{GCS_PREFIX}/qa-sft_{step:05d}.pt")
            wandb.save(ckpt_path)

    # Final summary
    if master:
        print(f"\n{'='*60}")
        print(f"🏁 TRAINING COMPLETE!")
        print(f"{'='*60}")
        print(f"   Best val_loss: {best_val_loss:.4f} at step {best_step}")
        print(f"   Best checkpoint: {OUT_DIR}/qa-sft_best.pt")
        print(f"   Total steps: {step}")
        print(f"{'='*60}\n")
        
        # Log final metrics
        wandb.log({
            "final/best_val_loss": best_val_loss,
            "final/best_step": best_step,
            "final/total_steps": step
        })

    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    train_qa_sft_ddp()


Overwriting sft-qa.py


In [21]:
!torchrun --nproc_per_node=2 dpo_fixed.py

W1124 09:37:01.103000 362 torch/distributed/run.py:792] 
W1124 09:37:01.103000 362 torch/distributed/run.py:792] *****************************************
W1124 09:37:01.103000 362 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1124 09:37:01.103000 362 torch/distributed/run.py:792] *****************************************
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` 

In [20]:
%%writefile dpo_fixed.py
# =========================================================
# DDP + UNIVERSALLY SAFE SDPA + SAVE EVERY EVAL + MULTI-SAMPLES
# =========================================================

import os, math, random, json, pickle
from contextlib import contextmanager
import torch
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import tiktoken
import wandb
from google.cloud import storage

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
SFT_CKPT = "sft_best_v1.pt"
REF_CKPT = "sft_best_v1.pt"

CUSTOM_JSONL = "/kaggle/working/dpo_gpt2_small.jsonl"

OUT_DIR = "dpo_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

LR = 5e-7
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 16
MAX_STEPS = 320
SEQ_LEN = 256
BETA = 0.1
WARMUP_STEPS = 30
WEIGHT_DECAY = 0.01
SEED = 1337

EVAL_EVERY = 40
VAL_ITERS = 20
UPLOAD_EVERY_EVAL = True

WANDB_PROJECT = "gpt2-dpo"
WANDB_RUN_NAME = "gpt2-custom-dpo"

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET = "gpt2-ultrafineweb"
GCS_PREFIX = "dpo_custom"

# ---------------------------------------------------------
# UNIVERSALLY SAFE SDPA (WORKS ON ALL GPUs)
# ---------------------------------------------------------
@contextmanager
def sdpa_safe_ctx(device_type: str):
    """
    Guaranteed safe on ALL PyTorch builds:
      - Kaggle T4 (sm75)
      - V100
      - A100/H100
    Uses only the parameters supported everywhere.
    """

    if device_type != "cuda":
        yield
        return

    # Try new API (no kwargs)
    if hasattr(torch.nn, "attention") and hasattr(torch.nn.attention, "sdpa_kernel"):
        try:
            with torch.nn.attention.sdpa_kernel():
                yield
                return
        except:
            pass

    # Fallback to legacy sdp_kernel (kwargs ALWAYS supported in PyTorch 2.x)
    try:
        with torch.backends.cuda.sdp_kernel(
            enable_flash=False,          # flash not stable on sm75
            enable_math=True,            # always available
            enable_mem_efficient=True    # safe
        ):
            yield
            return
    except:
        pass

    # Final fallback: no-op
    yield

# ---------------------------------------------------------
# DDP SETUP
# ---------------------------------------------------------
def ddp_setup():
    if "RANK" in os.environ:
        rank = int(os.environ["RANK"])
        world = int(os.environ["WORLD_SIZE"])
        local = int(os.environ["LOCAL_RANK"])
        dist.init_process_group(backend="nccl")
        torch.cuda.set_device(local)
        return True, rank, world, local, f"cuda:{local}"
    else:
        return False, 0, 1, 0, ("cuda" if torch.cuda.is_available() else "cpu")

def ddp_cleanup(ddp):
    if ddp and dist.is_initialized():
        dist.destroy_process_group()

def is_rank0(rank): return rank == 0

# ---------------------------------------------------------
# UPLOAD TO GCS
# ---------------------------------------------------------
def upload_to_gcs(local_path, remote):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET)
        blob = bucket.blob(remote)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET}/{remote}")
    except Exception as e:
        print("❌ Upload failed:", e)

# ---------------------------------------------------------
# MODEL LOADING
# ---------------------------------------------------------
from train_gpt2_v2 import GPT, GPTConfig
enc = tiktoken.get_encoding("gpt2")
PAD_ID = enc.eot_token

def load_gpt(path, device):
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
    except:
        torch.serialization.add_safe_globals([GPTConfig])
        ckpt = torch.load(path, map_location="cpu", weights_only=False)

    m = GPT(ckpt["config"])
    m.load_state_dict(ckpt["model"])
    return m.to(device), ckpt["config"]

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------
def load_custom_dpo(path):
    print("📥 Loading custom DPO dataset:", path)
    data = [json.loads(x) for x in open(path)]
    random.shuffle(data)
    split = int(len(data) * 0.1)
    return data[split:], data[:split]

# ---------------------------------------------------------
# BATCH PREP
# ---------------------------------------------------------
def prepare_batch(batch, device):
    ch = []
    rj = []
    for ex in batch:
        ch.append(torch.tensor(enc.encode(ex["chosen"])[:SEQ_LEN]))
        rj.append(torch.tensor(enc.encode(ex["rejected"])[:SEQ_LEN]))

    def pad(lst):
        mx = max(len(x) for x in lst)
        t = torch.full((len(lst), mx), PAD_ID, dtype=torch.long)
        m = torch.zeros(len(lst), mx)
        for i, seq in enumerate(lst):
            t[i, :len(seq)] = seq
            m[i, :len(seq)] = 1
        return t.to(device), m.to(device)
    return *pad(ch), *pad(rj)

# ---------------------------------------------------------
# LOG PROBS
# ---------------------------------------------------------
def get_logps(model, ids, mask, device_type):
    with sdpa_safe_ctx(device_type):
        with torch.amp.autocast(device_type=device_type):
            logits, _ = model(ids)

    logits = logits[:, :-1]
    labels = ids[:, 1:]
    lp = torch.nn.functional.log_softmax(logits, -1)
    tok_lp = lp.gather(2, labels.unsqueeze(2)).squeeze(2)
    m = mask[:, 1:]
    return (tok_lp * m).sum(1) / m.sum(1).clamp(min=1)

# ---------------------------------------------------------
# DPO LOSS
# ---------------------------------------------------------
def dpo_loss(pc, pr, rc, rr, beta=BETA):
    pi = pc - pr
    ref = rc - rr
    loss = -torch.nn.functional.logsigmoid(beta * (pi - ref))
    acc = (beta*(pc-rc) > beta*(pr-rr)).float().mean()
    return loss.mean(), acc

# ---------------------------------------------------------
# LR SCHEDULE
# ---------------------------------------------------------
def lr_for(step):
    if step < WARMUP_STEPS:
        return LR*(step+1)/WARMUP_STEPS
    p = (step-WARMUP_STEPS)/(MAX_STEPS-WARMUP_STEPS)
    return LR*0.5*(1+math.cos(math.pi*p))

# ---------------------------------------------------------
# GENERATE 3–4 SAMPLES
# ---------------------------------------------------------
def generate_samples(model, device, prompts):
    out = []
    for prompt in prompts:
        ids = torch.tensor([enc.encode(prompt)], dtype=torch.long).to(device)

        with torch.no_grad():
            for _ in range(80):
                logits, _ = model(ids)
                nxt = torch.argmax(logits[:, -1, :], dim=-1)
                ids = torch.cat([ids, nxt.unsqueeze(0)], dim=1)
                if nxt.item() == PAD_ID:
                    break

        out.append(enc.decode(ids[0].tolist()))
    return out

# ---------------------------------------------------------
# TRAIN LOOP
# ---------------------------------------------------------
def train():
    ddp, rank, world, local_rank, device = ddp_setup()
    device_type = "cuda" if "cuda" in device else "cpu"

    random.seed(SEED+rank)
    torch.manual_seed(SEED+rank)

    policy, config = load_gpt(SFT_CKPT, device)
    ref, _ = load_gpt(REF_CKPT, device)
    ref.eval()
    for p in ref.parameters(): p.requires_grad=False

    if ddp:
        policy = DDP(policy, device_ids=[local_rank], output_device=local_rank, broadcast_buffers=False)

    raw = policy.module if ddp else policy
    opt = raw.configure_optimizers(weight_decay=WEIGHT_DECAY, learning_rate=LR, device_type=device_type)
    scaler = amp.GradScaler(enabled=(device_type=="cuda"))

    train_data, val_data = load_custom_dpo(CUSTOM_JSONL)

    if is_rank0(rank):
        wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME)
        print("\n🚀 Starting CUSTOM DPO...\n")

    best = 0.0

    # LOOP
    for step in range(MAX_STEPS):
        policy.train()
        opt.zero_grad(set_to_none=True)

        loss_tot, acc_tot = 0.0, 0.0

        for _ in range(GRAD_ACCUM_STEPS):
            batch = random.sample(train_data, BATCH_SIZE)
            c_ids, c_m, r_ids, r_m = prepare_batch(batch, device)

            pc = get_logps(policy, c_ids, c_m, device_type)
            pr = get_logps(policy, r_ids, r_m, device_type)

            with torch.no_grad():
                rc = get_logps(ref, c_ids, c_m, device_type)
                rr = get_logps(ref, r_ids, r_m, device_type)

            loss, acc = dpo_loss(pc, pr, rc, rr)
            loss_tot += loss.item()
            acc_tot += acc.item()

            scaler.scale(loss/GRAD_ACCUM_STEPS).backward()

        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(raw.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        if is_rank0(rank) and step % 10 == 0:
            print(f"step {step} | loss {loss_tot:.4f} | acc {acc_tot/GRAD_ACCUM_STEPS:.3f}")
            wandb.log({"loss": loss_tot, "acc": acc_tot/GRAD_ACCUM_STEPS, "step": step})

        # ---------------- EVAL ----------------
        if step > 0 and step % EVAL_EVERY == 0:
            policy.eval()
            vloss, vacc = [], []

            with torch.no_grad():
                for _ in range(VAL_ITERS):
                    batch = random.sample(val_data, BATCH_SIZE)
                    c_ids, c_m, r_ids, r_m = prepare_batch(batch, device)

                    pc = get_logps(policy, c_ids, c_m, device_type)
                    pr = get_logps(policy, r_ids, r_m, device_type)
                    rc = get_logps(ref, c_ids, c_m, device_type)
                    rr = get_logps(ref, r_ids, r_m, device_type)

                    L, A = dpo_loss(pc, pr, rc, rr)
                    vloss.append(L.item()); vacc.append(A.item())

            val_loss = sum(vloss)/len(vloss)
            val_acc = sum(vacc)/len(vacc)

            if is_rank0(rank):
                print(f"📉 val_loss {val_loss:.4f} | val_acc {val_acc:.3f}")
                wandb.log({"val_loss": val_loss, "val_acc": val_acc, "step": step})

                # ----------- GENERATE 3–4 SAMPLES -----------
                prompts = [
                    "Hello, how are you?",
                    "Explain why self-discipline is important.",
                    "Write a motivational message for a student.",
                    "Describe what makes AI alignment challenging."
                ]

                samples = generate_samples(raw, device, prompts)

                print("\n📝 SAMPLES:")
                for i, s in enumerate(samples):
                    print(f"\n--- SAMPLE {i+1} ---\n{s}\n")

                wandb.log({f"sample_{i}": s for i, s in enumerate(samples)})

                # ---------------- SAVE EVERY EVAL ----------------
                step_path = f"{OUT_DIR}/dpo_step_{step:05d}.pt"
                torch.save({"model": raw.state_dict(), "config": config}, step_path)

                if UPLOAD_EVERY_EVAL:
                    upload_to_gcs(step_path, f"{GCS_PREFIX}/dpo_step_{step:05d}.pt")

                # ---------------- SAVE BEST ----------------
                if val_acc > best:
                    best = val_acc
                    best_path = f"{OUT_DIR}/dpo_best.pt"
                    torch.save({"model": raw.state_dict(), "config": config}, best_path)
                    upload_to_gcs(best_path, f"{GCS_PREFIX}/dpo_best.pt")
                    print("✨ NEW BEST MODEL!")

    # FINAL
    if is_rank0(rank):
        final_path = f"{OUT_DIR}/dpo_final.pt"
        torch.save({"model": raw.state_dict(), "config": config}, final_path)
        upload_to_gcs(final_path, f"{GCS_PREFIX}/dpo_final.pt")
        print("🏁 DONE")

    ddp_cleanup(ddp)


if __name__ == "__main__":
    train()


Overwriting dpo_fixed.py


In [12]:
!torchrun --nproc_per_node=2 dpo_train.py

W1124 07:28:13.587000 303 torch/distributed/run.py:792] 
W1124 07:28:13.587000 303 torch/distributed/run.py:792] *****************************************
W1124 07:28:13.587000 303 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1124 07:28:13.587000 303 torch/distributed/run.py:792] *****************************************
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` 

In [11]:
%%writefile dpo_train.py
import os, math, random, time
from tqdm import tqdm
import torch
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from datasets import load_dataset
import tiktoken
import wandb
from google.cloud import storage

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
# ✅ Use your V1 step 400 checkpoint (better than V2)
SFT_CKPT = "sft_checkpoints/sft_best.pt"   # Start from this
REF_CKPT = "sft_checkpoints/sft_best.pt"   # Keep this frozen as reference

OUT_DIR = "dpo_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# DPO hyperparams
LR = 5e-7              # ✅ Much lower for DPO
BATCH_SIZE = 2         
GRAD_ACCUM_STEPS = 16
MAX_STEPS = 300        # ✅ Reduced
SEQ_LEN = 512
BETA = 0.1             # ✅ Renamed from LAMBDA_DPO for clarity
WARMUP_STEPS = 30
WEIGHT_DECAY = 0.01    # ✅ Added
SEED = 1337

EVAL_EVERY = 50
SAVE_EVERY = 50

WANDB_PROJECT = "gpt2-dpo"
WANDB_RUN_NAME = "gpt2-dpo-fixed"

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET_NAME = "gpt2-ultrafineweb"
GCS_PREFIX = "dpo_checkpoints"

def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET_NAME)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET_NAME}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# ---------------------------------------------------------
# Model
# ---------------------------------------------------------
from train_gpt2_v2 import GPT, GPTConfig
enc = tiktoken.get_encoding("gpt2")
PAD_ID = enc.eot_token

def load_gpt(path, device):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    config: GPTConfig = ckpt["config"]
    model = GPT(config)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    return model, config

# ---------------------------------------------------------
# Load DPO Dataset - BETTER CHOICE
# ---------------------------------------------------------
def load_dpo_dataset():
    print("⬇️ Loading Anthropic HH-RLHF (better for instruction following)...")
    
    # Better dataset for instruction-following
    ds = load_dataset("Anthropic/hh-rlhf", split="train")
    ds = ds.shuffle(seed=SEED)
    
    # Sample for efficiency
    sample_size = min(50000, len(ds))
    ds = ds.select(range(sample_size))
    
    val_ds = load_dataset("Anthropic/hh-rlhf", split="test")
    val_ds = val_ds.select(range(min(1000, len(val_ds))))
    
    return ds, val_ds

# ---------------------------------------------------------
# ✅ FIXED: Proper log probability calculation
# ---------------------------------------------------------
def get_batch_log_probs(model, input_ids, labels, attention_mask=None):
    """
    Calculate log probabilities for a sequence.
    
    Args:
        model: GPT model
        input_ids: [batch, seq_len] input tokens
        labels: [batch, seq_len] target tokens (same as input_ids for causal LM)
        attention_mask: [batch, seq_len] mask for padding
    
    Returns:
        log_probs: [batch] sum of log probabilities per sequence
    """
    with torch.amp.autocast("cuda"):
        logits, _ = model(input_ids)  # [batch, seq_len, vocab_size]
    
    # Shift for next-token prediction
    logits = logits[:, :-1, :].contiguous()  # Remove last token
    labels = labels[:, 1:].contiguous()      # Remove first token
    
    # Calculate log probabilities
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    
    # Gather the log probs for the actual labels
    # Shape: [batch, seq_len-1]
    per_token_log_probs = torch.gather(
        log_probs, 
        dim=2, 
        index=labels.unsqueeze(2)
    ).squeeze(2)
    
    # Mask out padding tokens if provided
    if attention_mask is not None:
        mask = attention_mask[:, 1:].contiguous()
        per_token_log_probs = per_token_log_probs * mask
        
        # Sum over valid tokens only
        sequence_log_probs = per_token_log_probs.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    else:
        # Sum over all tokens
        sequence_log_probs = per_token_log_probs.mean(dim=1)
    
    return sequence_log_probs

# ---------------------------------------------------------
# ✅ FIXED: Tokenize with proper formatting
# ---------------------------------------------------------
def prepare_dpo_batch(batch, device, max_len=SEQ_LEN):
    """
    Prepare a batch for DPO training.
    Expected format: chosen and rejected are full conversations.
    """
    chosen_ids = []
    rejected_ids = []
    
    for ex in batch:
        # Anthropic HH-RLHF format: "chosen" and "rejected" are strings
        chosen_text = ex["chosen"]
        rejected_text = ex["rejected"]
        
        # Tokenize
        c_ids = enc.encode(chosen_text)[:max_len]
        r_ids = enc.encode(rejected_text)[:max_len]
        
        chosen_ids.append(torch.tensor(c_ids, dtype=torch.long))
        rejected_ids.append(torch.tensor(r_ids, dtype=torch.long))
    
    # Pad sequences
    def pad_sequences(sequences):
        max_len = max(len(s) for s in sequences)
        padded = torch.full((len(sequences), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros(len(sequences), max_len, dtype=torch.float)
        
        for i, seq in enumerate(sequences):
            padded[i, :len(seq)] = seq
            mask[i, :len(seq)] = 1.0
        
        return padded.to(device), mask.to(device)
    
    chosen_tokens, chosen_mask = pad_sequences(chosen_ids)
    rejected_tokens, rejected_mask = pad_sequences(rejected_ids)
    
    return chosen_tokens, chosen_mask, rejected_tokens, rejected_mask

# ---------------------------------------------------------
# ✅ FIXED: Proper DPO loss
# ---------------------------------------------------------
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=BETA):
    """
    DPO loss from the paper: https://arxiv.org/abs/2305.18290
    
    Loss = -log(sigmoid(beta * (log(pi_chosen/pi_ref_chosen) - log(pi_rejected/pi_ref_rejected))))
    """
    # Calculate log ratios
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    
    # DPO loss
    losses = -torch.nn.functional.logsigmoid(beta * (pi_logratios - ref_logratios))
    
    # Also calculate accuracy for monitoring
    rewards_chosen = beta * (policy_chosen_logps - ref_chosen_logps)
    rewards_rejected = beta * (policy_rejected_logps - ref_rejected_logps)
    accuracy = (rewards_chosen > rewards_rejected).float().mean()
    
    return losses.mean(), accuracy

# ---------------------------------------------------------
# ✅ FIXED: Cosine schedule
# ---------------------------------------------------------
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, (MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return LR * cosine

# ---------------------------------------------------------
# Main DPO Training
# ---------------------------------------------------------
def train_dpo():
    # DDP setup
    ddp = int(os.environ.get("RANK", -1)) != -1
    if ddp:
        dist.init_process_group("nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
        master = (rank == 0)
    else:
        rank, local_rank, world = 0, 0, 1
        device = DEVICE
        master = True

    torch.manual_seed(SEED + rank)
    random.seed(SEED + rank)

    # Load models
    if master:
        print("🔄 Loading policy model...")
    policy, config = load_gpt(SFT_CKPT, device)
    policy.train()

    if master:
        print("🔄 Loading frozen reference model...")
    ref_model, _ = load_gpt(REF_CKPT, device)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False

    if ddp:
        policy = DDP(policy, device_ids=[local_rank])
    raw_policy = policy.module if ddp else policy

    # Optimizer
    opt = raw_policy.configure_optimizers(
        weight_decay=WEIGHT_DECAY, 
        learning_rate=LR, 
        device_type=device
    )
    scaler = amp.GradScaler("cuda")

    # Dataset
    train_ds, val_ds = load_dpo_dataset()
    
    # Convert to list for sampling
    train_data = [train_ds[i] for i in range(len(train_ds))]
    val_data = [val_ds[i] for i in range(len(val_ds))]

    if master:
        wandb.init(
            project=WANDB_PROJECT, 
            name=WANDB_RUN_NAME,
            config={
                "lr": LR,
                "beta": BETA,
                "batch_size": BATCH_SIZE,
                "max_steps": MAX_STEPS,
                "weight_decay": WEIGHT_DECAY
            }
        )

    best_val_acc = 0.0
    
    if master:
        print("\n🚀 Starting DPO Training...\n")
    
    for step in range(MAX_STEPS):
        t0 = time.time()
        
        # Learning rate schedule
        lr = get_lr(step)
        for pg in opt.param_groups:
            pg["lr"] = lr
        
        # Training step
        policy.train()
        opt.zero_grad()
        
        loss_accum = 0.0
        acc_accum = 0.0
        
        for micro in range(GRAD_ACCUM_STEPS):
            # Sample batch
            batch = random.sample(train_data, BATCH_SIZE)
            chosen_ids, chosen_mask, rejected_ids, rejected_mask = prepare_dpo_batch(batch, device)
            
            # Get log probs from policy
            policy_chosen_logps = get_batch_log_probs(raw_policy, chosen_ids, chosen_ids, chosen_mask)
            policy_rejected_logps = get_batch_log_probs(raw_policy, rejected_ids, rejected_ids, rejected_mask)
            
            # Get log probs from reference (no grad)
            with torch.no_grad():
                ref_chosen_logps = get_batch_log_probs(ref_model, chosen_ids, chosen_ids, chosen_mask)
                ref_rejected_logps = get_batch_log_probs(ref_model, rejected_ids, rejected_ids, rejected_mask)
            
            # Calculate loss
            loss, acc = dpo_loss(
                policy_chosen_logps, policy_rejected_logps,
                ref_chosen_logps, ref_rejected_logps
            )
            
            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.item()
            acc_accum += acc.item() / GRAD_ACCUM_STEPS
            
            if ddp:
                policy.require_backward_grad_sync = (micro == GRAD_ACCUM_STEPS - 1)
            
            scaler.scale(loss).backward()
        
        # Optimizer step
        scaler.unscale_(opt)
        norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        
        if ddp:
            torch.cuda.synchronize()
        
        dt = time.time() - t0

        # Logging
        if master and step % 10 == 0:
            print(f"step {step:4d} | loss {loss_accum:.4f} | acc {acc_accum:.3f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt:.2f}s")
            wandb.log({
                "train/loss": loss_accum,
                "train/accuracy": acc_accum,
                "train/lr": lr,
                "train/grad_norm": norm,
                "step": step
            })

        # Validation
        if master and step % EVAL_EVERY == 0 and step > 0:
            policy.eval()
            val_losses = []
            val_accs = []
            
            with torch.no_grad():
                for _ in range(min(20, len(val_data) // BATCH_SIZE)):
                    batch = random.sample(val_data, BATCH_SIZE)
                    chosen_ids, chosen_mask, rejected_ids, rejected_mask = prepare_dpo_batch(batch, device)
                    
                    policy_chosen_logps = get_batch_log_probs(raw_policy, chosen_ids, chosen_ids, chosen_mask)
                    policy_rejected_logps = get_batch_log_probs(raw_policy, rejected_ids, rejected_ids, rejected_mask)
                    ref_chosen_logps = get_batch_log_probs(ref_model, chosen_ids, chosen_ids, chosen_mask)
                    ref_rejected_logps = get_batch_log_probs(ref_model, rejected_ids, rejected_ids, rejected_mask)
                    
                    loss, acc = dpo_loss(
                        policy_chosen_logps, policy_rejected_logps,
                        ref_chosen_logps, ref_rejected_logps
                    )
                    
                    val_losses.append(loss.item())
                    val_accs.append(acc.item())
            
            val_loss = sum(val_losses) / len(val_losses)
            val_acc = sum(val_accs) / len(val_accs)
            
            print(f"📉 val_loss: {val_loss:.4f} | val_acc: {val_acc:.3f}")
            wandb.log({"val/loss": val_loss, "val/accuracy": val_acc, "step": step})
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_path = os.path.join(OUT_DIR, "dpo_best.pt")
                torch.save({"model": raw_policy.state_dict(), "config": config, "step": step}, best_path)
                print(f"💾 ✨ NEW BEST! val_acc: {val_acc:.3f}")
                upload_to_gcs(best_path, f"{GCS_PREFIX}/dpo_best.pt")
            
            policy.train()

        # Save checkpoints
        if master and step % SAVE_EVERY == 0 and step > 0:
            ck = f"{OUT_DIR}/dpo_{step:05d}.pt"
            torch.save({"model": raw_policy.state_dict(), "config": config, "step": step}, ck)
            upload_to_gcs(ck, f"{GCS_PREFIX}/dpo_{step:05d}.pt")
            print(f"✅ Saved {ck}")

    # Final save
    if master:
        final = f"{OUT_DIR}/dpo_final.pt"
        torch.save({"model": raw_policy.state_dict(), "config": config, "step": MAX_STEPS}, final)
        upload_to_gcs(final, f"{GCS_PREFIX}/dpo_final.pt")
        print(f"\n🏁 Finished! Best val_acc: {best_val_acc:.3f}")

    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    train_dpo()

Writing dpo_train.py


In [ ]:
!torchrun --standalone --nproc_per_node=2 sft_ddp_v2.py

W1124 06:21:15.168000 174 torch/distributed/run.py:792] 
W1124 06:21:15.168000 174 torch/distributed/run.py:792] *****************************************
W1124 06:21:15.168000 174 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1124 06:21:15.168000 174 torch/distributed/run.py:792] *****************************************
[W1124 06:21:25.572203333 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1124 06:21:35.582656681 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is fiel

In [6]:
%%writefile sft_ddp_v2.py
import os, math, random, time
from tqdm import tqdm

import torch
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from datasets import load_dataset

import tiktoken
import wandb
from google.cloud import storage

# Flash Attention v1 ON (T4 supports v1)
from torch.backends.cuda import sdp_kernel
sdp_kernel(enable_flash=True, enable_mem_efficient=False, enable_math=False)

# -----------------------------
# Config - BETTER DATASETS
# -----------------------------
# 🎯 Choose ONE high-quality dataset:

# Option A: OpenOrca (high-quality, diverse instructions)
HF_DATASET = "Open-Orca/OpenOrca"
HF_SPLIT = "train"
USE_ORCA = True

# Option B: WizardLM Evol Instruct (complex reasoning)
# HF_DATASET = "WizardLM/WizardLM_evol_instruct_V2_196k"
# HF_SPLIT = "train"
# USE_ORCA = False

# Option C: Alpaca-GPT4 (higher quality than original Alpaca)
# HF_DATASET = "vicgalle/alpaca-gpt4"
# HF_SPLIT = "train"
# USE_ORCA = False

OUT_DIR = "sft_v2_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_CKPT = "sft_checkpoints/sft_best.pt"  # Continue from your step 400 checkpoint!
RESUME_CKPT = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ✅ Keep the working hyperparams from before
BATCH_SIZE = 4
SEQ_LEN = 1024
GRAD_ACCUM_STEPS = 32
LR = 8e-6              # Slightly lower for continued training
MIN_LR = 1e-7
WEIGHT_DECAY = 0.1
MAX_STEPS = 300        # Shorter run for refinement
WARMUP_STEPS = 30

EVAL_EVERY = 40
SAVE_EVERY = 40
SEED = 1337

# Early stopping
PATIENCE = 3
EARLY_STOP_ENABLED = True

# W&B
WANDB_PROJECT = "gpt2-sft-ultrafineweb"
WANDB_RUN_NAME = "sft-v2-orca-refined"

# GCS
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET_NAME = "gpt2-ultrafineweb"
GCS_PREFIX = "sft_v2_checkpoints"

def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET_NAME)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET_NAME}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# -----------------------------
# Import your model
# -----------------------------
from train_gpt2_v2 import GPT, GPTConfig

enc = tiktoken.get_encoding("gpt2")
EOS_ID = enc.eot_token

# -----------------------------
# Formatting (supports multiple dataset formats)
# -----------------------------
def format_orca(system, question, response):
    """Format for OpenOrca dataset"""
    return f"### System:\n{system}\n\n### User:\n{question}\n\n### Assistant:\n{response}"

def format_alpaca(inst, inp, out):
    """Format for Alpaca-style datasets"""
    inst = (inst or "").strip()
    inp = (inp or "").strip()
    out = (out or "").strip()

    if inp:
        text = (
            "### Instruction:\n" + inst + "\n\n"
            "### Input:\n" + inp + "\n\n"
            "### Response:\n" + out
        )
    else:
        text = (
            "### Instruction:\n" + inst + "\n\n"
            "### Response:\n" + out
        )
    return text

# -----------------------------
# Build token dataset
# -----------------------------
def build_token_dataset(master_process: bool):
    cache_file = "sft_v2_train_tokens.pt"
    cache_val = "sft_v2_val_tokens.pt"
    
    if os.path.exists(cache_file) and os.path.exists(cache_val):
        if master_process:
            print("✅ Token files already exist.")
        if dist.is_initialized():
            dist.barrier()
        return

    if master_process:
        print(f"⬇️ Loading {HF_DATASET}...")
        
        if USE_ORCA:
            # OpenOrca has different column names
            ds = load_dataset(HF_DATASET, split=HF_SPLIT)
            ds = ds.shuffle(seed=SEED)
            
            # Sample subset (OpenOrca is huge - 4.2M examples)
            sample_size = min(100000, len(ds))  # Use 100k examples
            ds = ds.select(range(sample_size))
            
            val_size = 1000
            val_ds = ds.select(range(val_size))
            train_ds = ds.select(range(val_size, len(ds)))
            
            def tokenize_orca(dataset, desc):
                all_ids = []
                for ex in tqdm(dataset, desc=desc):
                    system = ex.get("system_prompt", "")
                    question = ex.get("question", "")
                    response = ex.get("response", "")
                    
                    if not question or not response:
                        continue
                        
                    txt = format_orca(system, question, response)
                    ids = enc.encode(txt)
                    ids.append(EOS_ID)
                    all_ids.extend(ids)
                return torch.tensor(all_ids, dtype=torch.long)
            
            train_tokens = tokenize_orca(train_ds, "Tokenizing train")
            val_tokens = tokenize_orca(val_ds, "Tokenizing val")
        
        else:
            # Standard Alpaca format
            ds = load_dataset(HF_DATASET, split=HF_SPLIT)
            ds = ds.shuffle(seed=SEED)
            
            val_size = 1000
            val_ds = ds.select(range(val_size))
            train_ds = ds.select(range(val_size, len(ds)))
            
            def tokenize_ds(dataset, desc):
                all_ids = []
                for ex in tqdm(dataset, desc=desc):
                    txt = format_alpaca(
                        ex.get("instruction", ""),
                        ex.get("input", ""),
                        ex.get("output", "")
                    )
                    ids = enc.encode(txt)
                    ids.append(EOS_ID)
                    all_ids.extend(ids)
                return torch.tensor(all_ids, dtype=torch.long)
            
            train_tokens = tokenize_ds(train_ds, "Tokenizing train")
            val_tokens = tokenize_ds(val_ds, "Tokenizing val")

        torch.save(train_tokens, cache_file)
        torch.save(val_tokens, cache_val)
        print(f"✅ Saved {cache_file} and {cache_val}")

    if dist.is_initialized():
        dist.barrier()

# -----------------------------
# DDP TokenLoader
# -----------------------------
class TokenLoaderDDP:
    def __init__(self, tokens, batch_size, seq_len, process_rank, num_processes):
        self.tokens = tokens
        self.B = batch_size
        self.T = seq_len
        self.rank = process_rank
        self.world = num_processes
        self.pos = self.B * self.T * self.rank

    def next_batch(self):
        B, T = self.B, self.T
        need = B*T + 1
        if self.pos + need >= len(self.tokens):
            self.pos = self.B * self.T * self.rank
        buf = self.tokens[self.pos : self.pos + need]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        self.pos += B*T*self.world
        return x, y

# -----------------------------
# Load model
# -----------------------------
def load_model(device):
    ckpt_path = RESUME_CKPT if (RESUME_CKPT and os.path.exists(RESUME_CKPT)) else BASE_CKPT
    print(f"\n🔄 Loading ckpt: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config: GPTConfig = ckpt["config"]

    model = GPT(config)
    model.load_state_dict(ckpt["model"])
    model.to(device).train()

    start_step = int(ckpt.get("step", 0))
    
    # Reset for new training run
    if not RESUME_CKPT:
        start_step = 0

    print(f"✅ Model loaded. start_step={start_step}")
    return model, config, start_step

# -----------------------------
# Cosine LR schedule
# -----------------------------
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, (MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + cosine * (LR - MIN_LR)

# -----------------------------
# Eval + sample
# -----------------------------
@torch.no_grad()
def eval_loss(model, val_loader, device, batches=30):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with amp.autocast("cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses)

@torch.no_grad()
def sample_model(model, prompt, device, max_new=140, top_k=50):
    model.eval()
    ids = enc.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long)[None, :].to(device)

    for _ in range(max_new):
        logits, _ = model(x)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, k=top_k, dim=-1)
        ix = torch.multinomial(topk_probs, 1)
        next_id = torch.gather(topk_idx, -1, ix)
        x = torch.cat([x, next_id], dim=1)
        if next_id.item() == EOS_ID:
            break

    text = enc.decode(x[0].tolist())
    model.train()
    return text

# -----------------------------
# Main train
# -----------------------------
def train_sft_ddp():
    ddp = int(os.environ.get("RANK", -1)) != -1
    if ddp:
        dist.init_process_group("nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
        master = (rank == 0)
    else:
        rank, local_rank, world_size = 0, 0, 1
        device = DEVICE
        master = True

    random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    torch.cuda.manual_seed(SEED + rank)

    build_token_dataset(master)

    train_tokens = torch.load("sft_v2_train_tokens.pt")
    val_tokens = torch.load("sft_v2_val_tokens.pt")

    train_loader = TokenLoaderDDP(train_tokens, BATCH_SIZE, SEQ_LEN, rank, world_size)
    val_loader = TokenLoaderDDP(val_tokens, BATCH_SIZE, SEQ_LEN, rank, world_size)

    model, config, start_step = load_model(device)
    if ddp:
        model = DDP(model, device_ids=[local_rank])
    raw_model = model.module if ddp else model

    optimizer = raw_model.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=LR,
        device_type=device
    )
    scaler = amp.GradScaler(device="cuda")

    best_val_loss = float('inf')
    patience_counter = 0
    best_step = 0

    if master:
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=dict(
                dataset=HF_DATASET,
                base_ckpt=BASE_CKPT,
                batch_size=BATCH_SIZE,
                seq_len=SEQ_LEN,
                grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR,
                min_lr=MIN_LR,
                weight_decay=WEIGHT_DECAY,
                max_steps=MAX_STEPS,
                warmup_steps=WARMUP_STEPS,
                patience=PATIENCE,
                ddp_world_size=world_size,
            )
        )

    print("\n🚀 Starting SFT-V2 (refined training on better data)...")
    for step in range(start_step, MAX_STEPS):
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0

        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        for micro in range(GRAD_ACCUM_STEPS):
            x, y = train_loader.next_batch()
            x, y = x.to(device), y.to(device)

            with amp.autocast("cuda"):
                _, loss = model(x, y)

            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.item()

            if ddp:
                model.require_backward_grad_sync = (micro == GRAD_ACCUM_STEPS - 1)

            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()

        if ddp:
            loss_t = torch.tensor(loss_accum, device=device)
            dist.all_reduce(loss_t, op=dist.ReduceOp.AVG)
            loss_accum = loss_t.item()

        dt = time.time() - t0
        if master and step % 10 == 0:
            print(f"step {step:4d} | train_loss {loss_accum:.4f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt:.2f}s")
            wandb.log({"train/loss": loss_accum, "train/lr": lr, "train/grad_norm": norm, "step": step})

        if master and step % EVAL_EVERY == 0 and step > 0:
            vloss = eval_loss(raw_model, val_loader, device)
            print(f"📉 val_loss @ step {step}: {vloss:.4f}")
            wandb.log({"val/loss": vloss, "step": step})

            if vloss < best_val_loss:
                best_val_loss = vloss
                patience_counter = 0
                best_step = step
                
                best_path = os.path.join(OUT_DIR, "sft_v2_best.pt")
                torch.save({
                    "model": raw_model.state_dict(), 
                    "config": config, 
                    "step": step,
                    "val_loss": vloss
                }, best_path)
                print(f"💾 ✨ NEW BEST MODEL! val_loss: {vloss:.4f}")
                upload_to_gcs(best_path, f"{GCS_PREFIX}/sft_v2_best.pt")
            else:
                patience_counter += 1
                print(f"⚠️  Patience: {patience_counter}/{PATIENCE} (best: {best_val_loss:.4f} @ step {best_step})")
                
            if EARLY_STOP_ENABLED and patience_counter >= PATIENCE:
                print(f"\n🛑 EARLY STOPPING at step {step}")
                break

            # Test on multiple prompts
            test_prompts = [
                "### Instruction:\nExplain overfitting in machine learning.\n\n### Response:\n",
                "### System:\nYou are a helpful AI assistant.\n\n### User:\nWhat is the capital of France?\n\n### Assistant:\n",
                "### Instruction:\nWrite a Python function to calculate factorial.\n\n### Response:\n"
            ]
            
            for prompt in test_prompts[:1]:  # Show first one
                sample = sample_model(raw_model, prompt, device, max_new=100)
                print(f"\n🔮 Sample:\n{sample}\n")

        if master and step % SAVE_EVERY == 0 and step > 0:
            ckpt_path = os.path.join(OUT_DIR, f"sft_v2_{step:05d}.pt")
            torch.save({
                "model": raw_model.state_dict(), 
                "config": config, 
                "step": step
            }, ckpt_path)
            print(f"✅ saved {ckpt_path}")
            upload_to_gcs(ckpt_path, f"{GCS_PREFIX}/sft_v2_{step:05d}.pt")

    if master:
        print(f"\n{'='*60}")
        print(f"🏁 SFT-V2 Complete!")
        print(f"   Best val_loss: {best_val_loss:.4f} at step {best_step}")
        print(f"{'='*60}\n")

    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    train_sft_ddp()

Writing sft_ddp_v2.py


In [5]:
!torchrun --standalone --nproc_per_node=2 sft_ddp.py

W1124 04:51:03.331000 94 torch/distributed/run.py:792] 
W1124 04:51:03.331000 94 torch/distributed/run.py:792] *****************************************
W1124 04:51:03.331000 94 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1124 04:51:03.331000 94 torch/distributed/run.py:792] *****************************************
[W1124 04:51:13.734101808 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1124 04:51:23.744117302 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-sp

In [4]:
%%writefile sft_ddp.py
import os, math, random, time
from tqdm import tqdm

import torch
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from datasets import load_dataset

import tiktoken
import wandb
from google.cloud import storage

# Flash Attention v1 ON (T4 supports v1)
from torch.backends.cuda import sdp_kernel
sdp_kernel(enable_flash=True, enable_mem_efficient=False, enable_math=False)

# -----------------------------
# Config
# -----------------------------
HF_DATASET = "yahma/alpaca-cleaned"
HF_SPLIT   = "train"

OUT_DIR = "sft_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_CKPT   = "checkpoints/model_09535.pt"      # pretrained
RESUME_CKPT = None  # e.g. "sft_checkpoints/sft_00600.pt" if resuming

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ✅ IMPROVED Hyperparams - Fixed for overfitting
BATCH_SIZE = 4
SEQ_LEN = 1024
GRAD_ACCUM_STEPS = 32
LR = 1e-5              # ✅ REDUCED from 5e-5 (critical fix!)
MIN_LR = 1e-7          # ✅ Adjusted proportionally
WEIGHT_DECAY = 0.1
MAX_STEPS = 500        # ✅ REDUCED from 2000 (stop earlier!)
WARMUP_STEPS = 50      # ✅ REDUCED from 100 (shorter warmup)

EVAL_EVERY = 50        # ✅ Evaluate more frequently
SAVE_EVERY = 50        # ✅ Save more frequently
SEED = 1337

# ✅ EARLY STOPPING - New!
PATIENCE = 3           # Stop if val_loss doesn't improve for 3 evals
EARLY_STOP_ENABLED = True

# W&B
WANDB_PROJECT = "gpt2-sft-ultrafineweb"
WANDB_RUN_NAME = "sft-ddp-alpaca-cleaned-fixed"  # Updated name

# GCS
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET_NAME = "gpt2-ultrafineweb"
GCS_PREFIX = "sft_checkpoints"

def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET_NAME)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET_NAME}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# -----------------------------
# Import your model
# -----------------------------
from train_gpt2_v2 import GPT, GPTConfig

enc = tiktoken.get_encoding("gpt2")
EOS_ID = enc.eot_token

# -----------------------------
# Formatting
# -----------------------------
def format_example(inst, inp, out):
    inst = (inst or "").strip()
    inp  = (inp or "").strip()
    out  = (out or "").strip()

    if inp:
        text = (
            "### Instruction:\n" + inst + "\n\n"
            "### Input:\n" + inp + "\n\n"
            "### Response:\n" + out
        )
    else:
        text = (
            "### Instruction:\n" + inst + "\n\n"
            "### Response:\n" + out
        )
    return text

# -----------------------------
# Build token dataset (HF shuffle FIX)
# -----------------------------
def build_token_dataset(master_process: bool):
    if os.path.exists("sft_train_tokens.pt") and os.path.exists("sft_val_tokens.pt"):
        if master_process:
            print("✅ Token files already exist.")
        if dist.is_initialized():
            dist.barrier()
        return

    if master_process:
        print(f"⬇️ Loading {HF_DATASET}...")
        ds = load_dataset(HF_DATASET, split=HF_SPLIT)

        # ✅ FIX: HF shuffle (fast + safe)
        ds = ds.shuffle(seed=SEED)

        # val split
        val_size = 1000
        val_ds = ds.select(range(val_size))
        train_ds = ds.select(range(val_size, len(ds)))

        def tokenize_ds(dataset, desc):
            all_ids = []
            for ex in tqdm(dataset, desc=desc):
                txt = format_example(
                    ex.get("instruction",""),
                    ex.get("input",""),
                    ex.get("output","")
                )
                ids = enc.encode(txt)
                ids.append(EOS_ID)
                all_ids.extend(ids)
            return torch.tensor(all_ids, dtype=torch.long)

        train_tokens = tokenize_ds(train_ds, "Tokenizing train")
        val_tokens   = tokenize_ds(val_ds,   "Tokenizing val")

        torch.save(train_tokens, "sft_train_tokens.pt")
        torch.save(val_tokens, "sft_val_tokens.pt")
        print("✅ Saved sft_train_tokens.pt and sft_val_tokens.pt")

    if dist.is_initialized():
        dist.barrier()

# -----------------------------
# DDP TokenLoader
# -----------------------------
class TokenLoaderDDP:
    def __init__(self, tokens, batch_size, seq_len, process_rank, num_processes):
        self.tokens = tokens
        self.B = batch_size
        self.T = seq_len
        self.rank = process_rank
        self.world = num_processes
        self.pos = self.B * self.T * self.rank

    def next_batch(self):
        B, T = self.B, self.T
        need = B*T + 1
        if self.pos + need >= len(self.tokens):
            self.pos = self.B * self.T * self.rank
        buf = self.tokens[self.pos : self.pos + need]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        self.pos += B*T*self.world
        return x, y

# -----------------------------
# Load model (pretrain or resume)
# -----------------------------
def load_model(device):
    ckpt_path = RESUME_CKPT if (RESUME_CKPT and os.path.exists(RESUME_CKPT)) else BASE_CKPT
    print(f"\n🔄 Loading ckpt: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config: GPTConfig = ckpt["config"]

    model = GPT(config)
    model.load_state_dict(ckpt["model"])
    model.to(device).train()

    start_step = int(ckpt.get("step", 0))

    # Reset step counter if loading from base pretrained checkpoint
    if ckpt_path == BASE_CKPT:
        start_step = 0

    print(f"✅ Model loaded. start_step={start_step}")
    return model, config, start_step

# -----------------------------
# Cosine LR schedule
# -----------------------------
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, (MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + cosine * (LR - MIN_LR)

# -----------------------------
# Eval + sample
# -----------------------------
@torch.no_grad()
def eval_loss(model, val_loader, device, batches=30):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with amp.autocast("cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses)

@torch.no_grad()
def sample_model(model, prompt, device, max_new=140, top_k=50):
    model.eval()
    ids = enc.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long)[None, :].to(device)

    for _ in range(max_new):
        logits, _ = model(x)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, k=top_k, dim=-1)
        ix = torch.multinomial(topk_probs, 1)
        next_id = torch.gather(topk_idx, -1, ix)
        x = torch.cat([x, next_id], dim=1)
        if next_id.item() == EOS_ID:
            break

    text = enc.decode(x[0].tolist())
    model.train()
    return text

# -----------------------------
# Main train
# -----------------------------
def train_sft_ddp():
    ddp = int(os.environ.get("RANK", -1)) != -1
    if ddp:
        dist.init_process_group("nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
        master = (rank == 0)
    else:
        rank, local_rank, world_size = 0, 0, 1
        device = DEVICE
        master = True

    random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    torch.cuda.manual_seed(SEED + rank)

    build_token_dataset(master)

    train_tokens = torch.load("sft_train_tokens.pt")
    val_tokens   = torch.load("sft_val_tokens.pt")

    train_loader = TokenLoaderDDP(train_tokens, BATCH_SIZE, SEQ_LEN, rank, world_size)
    val_loader   = TokenLoaderDDP(val_tokens,   BATCH_SIZE, SEQ_LEN, rank, world_size)

    model, config, start_step = load_model(device)
    if ddp:
        model = DDP(model, device_ids=[local_rank])
    raw_model = model.module if ddp else model

    optimizer = raw_model.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=LR,
        device_type=device
    )
    scaler = amp.GradScaler(device="cuda")

    # ✅ Early stopping variables
    best_val_loss = float('inf')
    patience_counter = 0
    best_step = 0

    if master:
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=dict(
                dataset=HF_DATASET,
                base_ckpt=BASE_CKPT,
                resume_ckpt=RESUME_CKPT,
                batch_size=BATCH_SIZE,
                seq_len=SEQ_LEN,
                grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR,
                min_lr=MIN_LR,
                weight_decay=WEIGHT_DECAY,
                max_steps=MAX_STEPS,
                warmup_steps=WARMUP_STEPS,
                patience=PATIENCE,
                early_stop_enabled=EARLY_STOP_ENABLED,
                ddp_world_size=world_size,
                flash_attention=True,
            )
        )

    print("\n🚀 Starting SFT-DDP (alpaca-cleaned, anti-overfit)...")
    for step in range(start_step, MAX_STEPS):
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0

        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        for micro in range(GRAD_ACCUM_STEPS):
            x, y = train_loader.next_batch()
            x, y = x.to(device), y.to(device)

            with amp.autocast("cuda"):
                _, loss = model(x, y)

            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.item()

            if ddp:
                model.require_backward_grad_sync = (micro == GRAD_ACCUM_STEPS - 1)

            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()

        if ddp:
            loss_t = torch.tensor(loss_accum, device=device)
            dist.all_reduce(loss_t, op=dist.ReduceOp.AVG)
            loss_accum = loss_t.item()

        dt = time.time() - t0
        if master and step % 10 == 0:
            print(f"step {step:4d} | train_loss {loss_accum:.4f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt:.2f}s")
            wandb.log({"train/loss": loss_accum, "train/lr": lr, "train/grad_norm": norm, "step": step})

        # ✅ Evaluation with early stopping
        if master and step % EVAL_EVERY == 0 and step > 0:
            vloss = eval_loss(raw_model, val_loader, device)
            print(f"📉 val_loss @ step {step}: {vloss:.4f}")
            wandb.log({"val/loss": vloss, "step": step})

            # ✅ Early stopping logic
            if vloss < best_val_loss:
                best_val_loss = vloss
                patience_counter = 0
                best_step = step
                
                # Save best model
                best_path = os.path.join(OUT_DIR, "sft_best.pt")
                torch.save({
                    "model": raw_model.state_dict(), 
                    "config": config, 
                    "step": step,
                    "val_loss": vloss
                }, best_path)
                print(f"💾 ✨ NEW BEST MODEL! val_loss: {vloss:.4f}")
                upload_to_gcs(best_path, f"{GCS_PREFIX}/sft_best.pt")
                wandb.log({"best_val_loss": vloss, "best_step": step})
            else:
                patience_counter += 1
                print(f"⚠️  Val loss did not improve. Patience: {patience_counter}/{PATIENCE} (best: {best_val_loss:.4f} @ step {best_step})")
                wandb.log({"patience_counter": patience_counter, "step": step})
                
            if EARLY_STOP_ENABLED and patience_counter >= PATIENCE:
                print(f"\n🛑 EARLY STOPPING at step {step}")
                print(f"   Best val_loss: {best_val_loss:.4f} at step {best_step}")
                break

            # Sample generation
            prompt = "### Instruction:\nExplain overfitting in ML.\n\n### Response:\n"
            sample = sample_model(raw_model, prompt, device)
            print("\n🔮 sample:\n", sample[:1400], "\n")
            wandb.log({"sample": sample, "step": step})

        # Regular checkpoint saving
        if master and step % SAVE_EVERY == 0 and step > 0:
            ckpt_path = os.path.join(OUT_DIR, f"sft_{step:05d}.pt")
            torch.save({
                "model": raw_model.state_dict(), 
                "config": config, 
                "step": step
            }, ckpt_path)
            print(f"✅ saved {ckpt_path}")
            upload_to_gcs(ckpt_path, f"{GCS_PREFIX}/sft_{step:05d}.pt")
            wandb.save(ckpt_path)

    # Final summary
    if master:
        print(f"\n{'='*60}")
        print(f"🏁 Training Complete!")
        print(f"   Best val_loss: {best_val_loss:.4f} at step {best_step}")
        print(f"   Best checkpoint: {OUT_DIR}/sft_best.pt")
        print(f"{'='*60}\n")
        
        # Log final metrics
        wandb.log({
            "final/best_val_loss": best_val_loss,
            "final/best_step": best_step,
            "final/total_steps": step
        })

    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    train_sft_ddp()

Writing sft_ddp.py


In [19]:
!torchrun --standalone --nproc_per_node=2 sft_ddp.py

W1123 04:31:06.613000 278 torch/distributed/run.py:792] 
W1123 04:31:06.613000 278 torch/distributed/run.py:792] *****************************************
W1123 04:31:06.613000 278 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1123 04:31:06.613000 278 torch/distributed/run.py:792] *****************************************
[W1123 04:31:16.301204407 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1123 04:31:26.311722416 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is fiel

In [32]:
!torchrun --standalone --nproc_per_node=2 sft_ddp.py

W1123 06:19:42.381000 403 torch/distributed/run.py:792] 
W1123 06:19:42.381000 403 torch/distributed/run.py:792] *****************************************
W1123 06:19:42.381000 403 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1123 06:19:42.381000 403 torch/distributed/run.py:792] *****************************************
[W1123 06:19:52.953655138 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1123 06:20:02.961391881 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is fiel

In [48]:
!torchrun --nproc_per_node=2 dpo_ddp.py

W1123 08:47:35.728000 837 torch/distributed/run.py:792] 
W1123 08:47:35.728000 837 torch/distributed/run.py:792] *****************************************
W1123 08:47:35.728000 837 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1123 08:47:35.728000 837 torch/distributed/run.py:792] *****************************************
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` 

In [47]:
%%writefile dpo_ddp.py
import os, math, time, random, json
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn.functional as F
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler

import tiktoken
from datasets import load_dataset
from google.cloud import storage
import wandb

# -----------------------------
# Flash Attention v1 ON (T4 supports v1)
# -----------------------------
from torch.backends.cuda import sdp_kernel
sdp_kernel(enable_flash=True, enable_mem_efficient=False, enable_math=False)

# -----------------------------
# Config
# -----------------------------
DPO_DATASET = "HuggingFaceH4/ultrafeedback_binarized"
TRAIN_SPLIT = "train_prefs"
VAL_SPLIT   = "test_prefs"   # dataset provides test

# Choose best SFT checkpoint as init
DPO_START_CKPT = "sft_checkpoints/sft_00600.pt"  # change if needed

OUT_DIR = "dpo_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Hyperparams
BATCH_SIZE = 2
SEQ_LEN = 1024
GRAD_ACCUM_STEPS = 32

# ===== FIX: stabilize LR + warmup to avoid NaN / inf grads =====
LR = 1e-5
MIN_LR = 5e-7
WARMUP_STEPS = 200

WEIGHT_DECAY = 0.0
MAX_STEPS = 1200
BETA = 0.1  # DPO beta (0.05–0.2 typical)

EVAL_EVERY = 200
SAVE_EVERY = 200
SEED = 1337

# W&B
WANDB_PROJECT = "gpt2-dpo-ultrafineweb"
WANDB_RUN_NAME = "dpo-ddp-ultrafeedback"

# GCS
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET_NAME = "gpt2-ultrafineweb"
GCS_PREFIX = "dpo_checkpoints"

def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET_NAME)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET_NAME}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# -----------------------------
# Import your model
# -----------------------------
from train_gpt2_v2 import GPT, GPTConfig  # your exact model

enc = tiktoken.get_encoding("gpt2")
EOS_ID = enc.eot_token

# ============================================================
# FIX 1: UltraFeedback fields may be list/dict, not pure string
# ============================================================
def to_text(x):
    """Robustly convert UltraFeedback prompt/chosen/rejected to plain text."""
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, dict):
        if "content" in x and isinstance(x["content"], str):
            return x["content"]
        if "text" in x and isinstance(x["text"], str):
            return x["text"]
        return json.dumps(x, ensure_ascii=False)
    if isinstance(x, list):
        parts = []
        for item in x:
            parts.append(to_text(item))
        return "\n".join([p for p in parts if p])
    return str(x)

# -----------------------------
# Helpers: tokenize + truncate
# -----------------------------
def tokenize_prompt_and_response(prompt: str, resp: str, max_len=SEQ_LEN):
    prompt = to_text(prompt)
    resp   = to_text(resp)

    p_ids = enc.encode(prompt)
    r_ids = enc.encode(resp) + [EOS_ID]

    # FIRST truncate response to max_len - 16 (reserve minimal prompt room)
    if len(r_ids) > max_len - 16:
        r_ids = r_ids[: max_len - 16]

    # THEN truncate prompt to fill remaining space
    remain = max_len - len(r_ids)
    if remain < 16:
        remain = 16
    p_ids = p_ids[:remain]

    return p_ids, r_ids

# -----------------------------
# DPO Dataset
# ultrafeedback_binarized fields:
#  - prompt
#  - chosen
#  - rejected
# -----------------------------
class DPODataset(Dataset):
    def __init__(self, hf_ds):
        self.data = hf_ds

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        prompt = ex["prompt"]
        chosen = ex["chosen"]
        rejected = ex["rejected"]

        prompt_t = to_text(prompt)
        chosen_t = to_text(chosen)
        rejected_t = to_text(rejected)

        p_ids, c_ids = tokenize_prompt_and_response(prompt_t, chosen_t)
        _,   r_ids = tokenize_prompt_and_response(prompt_t, rejected_t)

        return {
            "prompt_ids": torch.tensor(p_ids, dtype=torch.long),
            "chosen_ids": torch.tensor(c_ids, dtype=torch.long),
            "rejected_ids": torch.tensor(r_ids, dtype=torch.long),
        }

def collate_fn(batch: List[Dict]):
    def pad(seqs):
        maxlen = max(s.size(0) for s in seqs)
        out = torch.full((len(seqs), maxlen), fill_value=EOS_ID, dtype=torch.long)
        for i, s in enumerate(seqs):
            out[i, :s.size(0)] = s
        return out

    prompt = pad([b["prompt_ids"] for b in batch])
    chosen = pad([b["chosen_ids"] for b in batch])
    rejected = pad([b["rejected_ids"] for b in batch])
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

# -----------------------------
# Load SFT policy + frozen reference
# -----------------------------
def load_policy_and_ref(device):
    print(f"\n🔄 Loading SFT ckpt for DPO: {DPO_START_CKPT}")
    ckpt = torch.load(DPO_START_CKPT, map_location="cpu", weights_only=False)
    config: GPTConfig = ckpt["config"]

    policy = GPT(config)
    policy.load_state_dict(ckpt["model"])
    policy.to(device).train()

    ref = GPT(config)
    ref.load_state_dict(ckpt["model"])
    ref.to(device).eval()
    for p in ref.parameters():
        p.requires_grad_(False)

    start_step = int(ckpt.get("step", 0))
    print("✅ Loaded policy + reference.")
    return policy, ref, config, start_step

# -----------------------------
# Cosine LR schedule
# -----------------------------
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, (MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + cosine * (LR - MIN_LR)

# -----------------------------
# Log-prob helper
# returns sum logp of completion tokens given prompt
# -----------------------------
def sequence_logprob(model, prompt_ids, completion_ids):
    block = model.module.config.block_size if isinstance(model, DDP) else model.config.block_size

    max_comp = block - prompt_ids.size(1)
    if max_comp < 1:
        max_comp = 1
    if completion_ids.size(1) > max_comp:
        completion_ids = completion_ids[:, :max_comp]

    x = torch.cat([prompt_ids, completion_ids[:, :-1]], dim=1)
    targets = completion_ids

    logits, _ = model(x)
    comp_logits = logits[:, -targets.size(1):, :]
    logprobs = torch.log_softmax(comp_logits, dim=-1)
    tok_logp = torch.gather(logprobs, -1, targets.unsqueeze(-1)).squeeze(-1)
    return tok_logp.sum(dim=1)

# -----------------------------
# Eval loss on val prefs
# -----------------------------
@torch.no_grad()
def eval_dpo_loss(policy, ref, loader, device, batches=50):
    policy.eval()
    losses = []
    for i, batch in enumerate(loader):
        if i >= batches: break
        p = batch["prompt"].to(device)
        c = batch["chosen"].to(device)
        r = batch["rejected"].to(device)

        with amp.autocast("cuda"):
            lp_c = sequence_logprob(policy, p, c)
            lp_r = sequence_logprob(policy, p, r)
            lr_c = sequence_logprob(ref, p, c)
            lr_r = sequence_logprob(ref, p, r)

            diff = (lp_c - lp_r) - (lr_c - lr_r)

            # ===== FIX: clamp diff to stop exploding grads =====
            diff = diff.clamp(min=-10, max=10)

            loss = -F.logsigmoid(BETA * diff).mean()

        losses.append(loss.item())

    policy.train()
    return sum(losses)/len(losses)

# -----------------------------
# Sample generation
# -----------------------------
@torch.no_grad()
def sample_model(model, prompt, device, max_new=120, top_k=50):
    model.eval()
    ids = enc.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long)[None, :].to(device)

    for _ in range(max_new):
        logits, _ = model(x)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, k=top_k, dim=-1)
        ix = torch.multinomial(topk_probs, 1)
        next_id = torch.gather(topk_idx, -1, ix)
        x = torch.cat([x, next_id], dim=1)

    text = enc.decode(x[0].tolist())
    model.train()
    return text

# -----------------------------
# Main DPO train (DDP)
# -----------------------------
def train_dpo_ddp():
    ddp = int(os.environ.get("RANK", -1)) != -1
    if ddp:
        dist.init_process_group("nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
        master = (rank == 0)
    else:
        rank, local_rank, world_size = 0, 0, 1
        device = DEVICE
        master = True

    random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    torch.cuda.manual_seed(SEED + rank)

    if master: print(f"⬇️ Loading DPO dataset: {DPO_DATASET}")
    train_ds = load_dataset(DPO_DATASET, split=TRAIN_SPLIT)
    val_ds   = load_dataset(DPO_DATASET, split=VAL_SPLIT)

    train_set = DPODataset(train_ds)
    val_set   = DPODataset(val_ds)

    train_sampler = DistributedSampler(train_set, num_replicas=world_size, rank=rank, shuffle=True) if ddp else None
    val_sampler   = DistributedSampler(val_set,   num_replicas=world_size, rank=rank, shuffle=False) if ddp else None

    train_loader = DataLoader(
        train_set, batch_size=BATCH_SIZE, sampler=train_sampler,
        shuffle=(train_sampler is None), collate_fn=collate_fn, drop_last=True
    )
    val_loader = DataLoader(
        val_set, batch_size=BATCH_SIZE, sampler=val_sampler,
        shuffle=False, collate_fn=collate_fn, drop_last=True
    )

    policy, ref, config, start_step = load_policy_and_ref(device)

    if ddp:
        policy = DDP(policy, device_ids=[local_rank])
    raw_policy = policy.module if ddp else policy

    optimizer = raw_policy.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=LR,
        device_type=device
    )
    scaler = amp.GradScaler(device="cuda")

    if master:
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=dict(
                dpo_dataset=DPO_DATASET,
                start_ckpt=DPO_START_CKPT,
                batch_size=BATCH_SIZE,
                seq_len=SEQ_LEN,
                grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR,
                min_lr=MIN_LR,
                warmup_steps=WARMUP_STEPS,
                max_steps=MAX_STEPS,
                beta=BETA,
                ddp_world_size=world_size,
                flash_attention=True,
            )
        )

    print("\n🚀 Starting DPO-DDP ...")

    it = iter(train_loader)

    for step in range(0, MAX_STEPS):
        if ddp: train_sampler.set_epoch(step)
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0

        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        for micro in range(GRAD_ACCUM_STEPS):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(train_loader)
                batch = next(it)

            p = batch["prompt"].to(device)
            c = batch["chosen"].to(device)
            r = batch["rejected"].to(device)

            with amp.autocast("cuda"):
                lp_c = sequence_logprob(policy, p, c)
                lp_r = sequence_logprob(policy, p, r)
                lr_c = sequence_logprob(ref, p, c)
                lr_r = sequence_logprob(ref, p, r)

                diff = (lp_c - lp_r) - (lr_c - lr_r)

                # ===== FIX: clamp diff to stop exploding grads =====
                diff = diff.clamp(min=-10, max=10)

                loss = -F.logsigmoid(BETA * diff).mean()

            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.item()

            if ddp:
                policy.require_backward_grad_sync = (micro == GRAD_ACCUM_STEPS - 1)

            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        # ===== FIX: tighter grad clip for DPO stability =====
        norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)

        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()

        if ddp:
            loss_t = torch.tensor(loss_accum, device=device)
            dist.all_reduce(loss_t, op=dist.ReduceOp.AVG)
            loss_accum = loss_t.item()

        dt = time.time() - t0
        if master and step % 10 == 0:
            print(f"step {step:4d} | dpo_loss {loss_accum:.4f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt:.2f}s")
            wandb.log({"train/dpo_loss": loss_accum, "train/lr": lr, "train/grad_norm": float(norm), "step": step})

        if master and step % EVAL_EVERY == 0 and step > 0:
            vloss = eval_dpo_loss(raw_policy, ref, val_loader, device)
            print(f"📉 val_dpo_loss @ step {step}: {vloss:.4f}")
            wandb.log({"val/dpo_loss": vloss, "step": step})

            prompt = "### Instruction:\nExplain overfitting in ML.\n\n### Response:\n"
            sample = sample_model(raw_policy, prompt, device)
            print("\n🔮 sample:\n", sample[:1400], "\n")
            wandb.log({"sample": sample, "step": step})

        if master and step % SAVE_EVERY == 0 and step > 0:
            ckpt_path = os.path.join(OUT_DIR, f"dpo_{step:05d}.pt")
            torch.save({"model": raw_policy.state_dict(), "config": config, "step": step}, ckpt_path)
            print(f"✅ saved {ckpt_path}")
            upload_to_gcs(ckpt_path, f"{GCS_PREFIX}/dpo_{step:05d}.pt")
            wandb.save(ckpt_path)

    if master:
        final_path = os.path.join(OUT_DIR, "dpo_final.pt")
        torch.save({"model": raw_policy.state_dict(), "config": config, "step": MAX_STEPS}, final_path)
        print(f"\n🏁 Finished! Final DPO checkpoint: {final_path}")
        upload_to_gcs(final_path, f"{GCS_PREFIX}/dpo_final.pt")
        wandb.save(final_path)

    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    train_dpo_ddp()


Overwriting dpo_ddp.py


In [31]:
%%writefile sft_ddp.py
import os, json, math, random, requests, time
from tqdm import tqdm

import torch
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from datasets import load_dataset

import tiktoken
import wandb
from google.cloud import storage

# Flash Attention v1 ON (T4 supports v1)
from torch.backends.cuda import sdp_kernel
sdp_kernel(enable_flash=True, enable_mem_efficient=False, enable_math=False)

# -----------------------------
# Config
# -----------------------------
DATA_URL  = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
DATA_PATH = "alpaca_data.json"

# extra SFT datasets (HF)
USE_LIMA = False
USE_ULTRACHAT = False
ULTRACHAT_MAX_EXAMPLES = 20000  # keep small for Kaggle

OUT_DIR = "sft_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_CKPT   = "checkpoints/model_09535.pt"      # pretrained
RESUME_CKPT = "sft_checkpoints/sft_00500.pt"    # your SFT 500 (set None to ignore)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Hyperparams
BATCH_SIZE = 4
SEQ_LEN = 1024
GRAD_ACCUM_STEPS = 32
LR = 5e-5
MIN_LR = 1e-6
WEIGHT_DECAY = 0.1
MAX_STEPS = 2000
WARMUP_STEPS = 100
EVAL_EVERY = 200
SAVE_EVERY = 200

SEED = 1337

# W&B
WANDB_PROJECT = "gpt2-sft-ultrafineweb"
WANDB_RUN_NAME = "sft-ddp-1024-cosine-resume"

# GCS
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET_NAME = "gpt2-ultrafineweb"
GCS_PREFIX = "sft_checkpoints"

def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET_NAME)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET_NAME}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# -----------------------------
# Import your model
# -----------------------------
from train_gpt2_v2 import GPT, GPTConfig

enc = tiktoken.get_encoding("gpt2")
EOS_ID = enc.eot_token

# -----------------------------
# Dataset helpers
# -----------------------------
def download_alpaca():
    if os.path.exists(DATA_PATH):
        print(f"✅ Dataset already exists: {DATA_PATH}")
        return
    print("⬇️ Downloading Alpaca 52K...")
    r = requests.get(DATA_URL)
    r.raise_for_status()
    with open(DATA_PATH, "w") as f:
        f.write(r.text)
    print("✅ Downloaded Alpaca.")

def format_example(inst, inp, out):
    inst = (inst or "").strip()
    inp  = (inp or "").strip()
    out  = (out or "").strip()

    if inp:
        text = (
            "### Instruction:\n" + inst + "\n\n"
            "### Input:\n" + inp + "\n\n"
            "### Response:\n" + out
        )
    else:
        text = (
            "### Instruction:\n" + inst + "\n\n"
            "### Response:\n" + out
        )
    return text

def load_extra_datasets():
    extras = []

    try:
        from datasets import load_dataset
    except Exception:
        print("⚠️ datasets lib not found. Run: pip install -q datasets")
        return extras

    if USE_LIMA:
        print("⬇️ Loading LIMA...")
        # lima = load_dataset("GAIR/lima", split="train")
        # for ex in lima:
            # extras.append({
                # "instruction": ex["instruction"],
                # "input": "",
                # "output": ex["response"]
            # })

    if USE_ULTRACHAT:
        print("⬇️ Loading UltraChat slice...")
        uc = load_dataset("HuggingFaceH4/ultrachat_200k", split="train")
        # each item has "messages": list of turns
        count = 0
        for ex in uc:
            msgs = ex.get("messages", [])
            if len(msgs) < 2:
                continue
            user_msg = msgs[0]["content"]
            asst_msg = msgs[1]["content"]
            extras.append({
                "instruction": user_msg,
                "input": "",
                "output": asst_msg
            })
            count += 1
            if count >= ULTRACHAT_MAX_EXAMPLES:
                break

    print(f"✅ Loaded extra examples: {len(extras)}")
    return extras

def build_token_dataset_if_needed(master_process: bool):
    if os.path.exists("sft_train_tokens.pt") and os.path.exists("sft_val_tokens.pt"):
        if master_process:
            print("✅ Token files already exist.")
        if dist.is_initialized():
            dist.barrier()
        return

    if master_process:
        download_alpaca()
        alpaca = json.load(open(DATA_PATH))
        random.shuffle(alpaca)

        extras = load_extra_datasets()
        all_data = alpaca + extras
        random.shuffle(all_data)

        # val split
        val_size = 1000
        val_data = all_data[:val_size]
        train_data = all_data[val_size:]

        def tokenize_list(ex_list, desc):
            all_ids = []
            for ex in tqdm(ex_list, desc=desc):
                txt = format_example(ex["instruction"], ex.get("input",""), ex["output"])
                ids = enc.encode(txt)
                ids.append(EOS_ID)
                all_ids.extend(ids)
            return torch.tensor(all_ids, dtype=torch.long)

        train_tokens = tokenize_list(train_data, "Tokenizing train")
        val_tokens   = tokenize_list(val_data, "Tokenizing val")

        torch.save(train_tokens, "sft_train_tokens.pt")
        torch.save(val_tokens, "sft_val_tokens.pt")
        print("✅ Saved sft_train_tokens.pt and sft_val_tokens.pt")

    if dist.is_initialized():
        dist.barrier()

# -----------------------------
# DDP TokenLoader
# -----------------------------
class TokenLoaderDDP:
    def __init__(self, tokens, batch_size, seq_len, process_rank, num_processes):
        self.tokens = tokens
        self.B = batch_size
        self.T = seq_len
        self.rank = process_rank
        self.world = num_processes
        self.pos = self.B * self.T * self.rank

    def next_batch(self):
        B, T = self.B, self.T
        need = B*T + 1
        if self.pos + need >= len(self.tokens):
            self.pos = self.B * self.T * self.rank
        buf = self.tokens[self.pos : self.pos + need]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        self.pos += B*T*self.world
        return x, y

# -----------------------------
# Load model (pretrain or resume)
# -----------------------------
def load_model(device):
    ckpt_path = RESUME_CKPT if (RESUME_CKPT and os.path.exists(RESUME_CKPT)) else BASE_CKPT
    print(f"\n🔄 Loading ckpt: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config: GPTConfig = ckpt["config"]

    model = GPT(config)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    model.train()

    start_step = int(ckpt.get("step", 0))
    print(f"✅ Model loaded. start_step={start_step}")
    return model, config, start_step

# -----------------------------
# Cosine LR schedule
# -----------------------------
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    # cosine decay from LR -> MIN_LR
    progress = (step - WARMUP_STEPS) / max(1, (MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + cosine * (LR - MIN_LR)

# -----------------------------
# Eval + sample
# -----------------------------
@torch.no_grad()
def eval_loss(model, val_loader, device, batches=30):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with amp.autocast("cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses)

@torch.no_grad()
def sample_model(model, prompt, device, max_new=120, top_k=50):
    model.eval()
    ids = enc.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long)[None, :].to(device)

    for _ in range(max_new):
        logits, _ = model(x)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, k=top_k, dim=-1)
        ix = torch.multinomial(topk_probs, 1)
        next_id = torch.gather(topk_idx, -1, ix)
        x = torch.cat([x, next_id], dim=1)

    text = enc.decode(x[0].tolist())
    model.train()
    return text

# -----------------------------
# Main train
# -----------------------------
def train_sft_ddp():
    ddp = int(os.environ.get("RANK", -1)) != -1
    if ddp:
        dist.init_process_group("nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
        master_process = (rank == 0)
    else:
        rank, local_rank, world_size = 0, 0, 1
        device = DEVICE
        master_process = True

    random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    torch.cuda.manual_seed(SEED + rank)

    build_token_dataset_if_needed(master_process)

    train_tokens = torch.load("sft_train_tokens.pt")
    val_tokens   = torch.load("sft_val_tokens.pt")

    train_loader = TokenLoaderDDP(train_tokens, BATCH_SIZE, SEQ_LEN, rank, world_size)
    val_loader   = TokenLoaderDDP(val_tokens,   BATCH_SIZE, SEQ_LEN, rank, world_size)

    model, config, start_step = load_model(device)
    if ddp:
        model = DDP(model, device_ids=[local_rank])
    raw_model = model.module if ddp else model

    optimizer = raw_model.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=LR,
        device_type=device
    )
    scaler = amp.GradScaler(device="cuda")

    if master_process:
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=dict(
                base_ckpt=BASE_CKPT,
                resume_ckpt=RESUME_CKPT,
                batch_size=BATCH_SIZE,
                seq_len=SEQ_LEN,
                grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR,
                min_lr=MIN_LR,
                weight_decay=WEIGHT_DECAY,
                max_steps=MAX_STEPS,
                warmup_steps=WARMUP_STEPS,
                ddp_world_size=world_size,
                flash_attention=True,
                use_lima=USE_LIMA,
                use_ultrachat=USE_ULTRACHAT,
                ultrachat_max=ULTRACHAT_MAX_EXAMPLES,
            )
        )

    print("\n🚀 Starting SFT-DDP (resume + cosine)...")
    for step in range(start_step, MAX_STEPS):
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0

        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        for micro in range(GRAD_ACCUM_STEPS):
            x, y = train_loader.next_batch()
            x, y = x.to(device), y.to(device)

            with amp.autocast("cuda"):
                _, loss = model(x, y)

            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.item()

            if ddp:
                model.require_backward_grad_sync = (micro == GRAD_ACCUM_STEPS - 1)

            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()

        if ddp:
            loss_t = torch.tensor(loss_accum, device=device)
            dist.all_reduce(loss_t, op=dist.ReduceOp.AVG)
            loss_accum = loss_t.item()

        dt = time.time() - t0
        if master_process and step % 10 == 0:
            print(f"step {step:4d} | train_loss {loss_accum:.4f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt:.2f}s")
            wandb.log({"train/loss": loss_accum, "train/lr": lr, "train/grad_norm": norm, "step": step})

        if master_process and step % EVAL_EVERY == 0 and step > start_step:
            vloss = eval_loss(raw_model, val_loader, device)
            print(f"📉 val_loss @ step {step}: {vloss:.4f}")
            wandb.log({"val/loss": vloss, "step": step})

            prompt = "### Instruction:\nExplain overfitting in ML.\n\n### Response:\n"
            sample = sample_model(raw_model, prompt, device)
            print("\n🔮 sample:\n", sample[:1400], "\n")
            wandb.log({"sample": sample, "step": step})

        if master_process and step % SAVE_EVERY == 0 and step > start_step:
            ckpt_path = os.path.join(OUT_DIR, f"sft_{step:05d}.pt")
            torch.save({"model": raw_model.state_dict(), "config": config, "step": step}, ckpt_path)
            print(f"✅ saved {ckpt_path}")
            upload_to_gcs(ckpt_path, f"{GCS_PREFIX}/sft_{step:05d}.pt")
            wandb.save(ckpt_path)

    if master_process:
        final_path = os.path.join(OUT_DIR, "sft_final.pt")
        torch.save({"model": raw_model.state_dict(), "config": config, "step": MAX_STEPS}, final_path)
        print(f"\n🏁 Finished! Final SFT checkpoint: {final_path}")
        upload_to_gcs(final_path, f"{GCS_PREFIX}/sft_final.pt")
        wandb.save(final_path)

    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    train_sft_ddp()


Overwriting sft_ddp.py


In [26]:
!python sample_sft_500.py

/usr/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value Tr

In [25]:
%%writefile sample_sft_500.py
import torch
from train_gpt2_v2 import GPT, GPTConfig
import tiktoken

CHECKPOINT = "sft_00500.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

enc = tiktoken.get_encoding("gpt2")

def load_sft_model():
    print(f"🔄 Loading SFT checkpoint: {CHECKPOINT}")

    from torch.serialization import add_safe_globals
    from train_gpt2_v2 import GPTConfig
    add_safe_globals([GPTConfig])

    ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)

    config: GPTConfig = ckpt["config"]
    model = GPT(config)

    model.load_state_dict(ckpt["model"])
    model.to(DEVICE)
    model.eval()

    print("✅ Model loaded.")
    return model


@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, top_k=50):
    ids = enc.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long)[None, :].to(DEVICE)

    for _ in range(max_new_tokens):
        logits, _ = model(x)
        logits = logits[:, -1, :] / 1.0    # temperature=1
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, k=top_k)
        ix = torch.multinomial(topk_probs, 1)
        next_id = torch.gather(topk_idx, -1, ix)
        x = torch.cat([x, next_id], dim=1)

    return enc.decode(x[0].tolist())

if __name__ == "__main__":
    model = load_sft_model()

    prompts = [
        "### Instruction:\nExplain overfitting in machine learning.\n\n### Response:\n",
        "### Instruction:\nWrite a motivational message for a student learning AI.\n\n### Response:\n",
        "### Instruction:\nSolve: What is 25 * 4?\n\n### Response:\n",
        "### Instruction:\nDescribe what a transformer model is.\n\n### Response:\n",
        "### Instruction:\nGive 3 tips for mental health.\n\n### Response:\n"
    ]

    for p in prompts:
        print("🧠 Prompt:", p)
        print("➡️ Output:")
        print(generate(model, p))
        print("="*80)


Overwriting sample_sft_500.py


In [22]:
!python download_sft_500.py

🔄 Connecting to GCS...
⬇️ Downloading gs://gpt2-ultrafineweb/sft_checkpoints/sft_00500.pt
✅ Downloaded to sft_00500.pt


In [20]:
%%writefile download_sft_500.py
import os
from google.cloud import storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/kaggle/working/gpt2-bucket-key.json"

BUCKET_NAME = "gpt2-ultrafineweb"
BLOB_NAME = "sft_checkpoints/sft_00500.pt"
LOCAL_PATH = "sft_00500.pt"

def download():
    print("🔄 Connecting to GCS...")
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    blob = bucket.blob(BLOB_NAME)

    print(f"⬇️ Downloading gs://{BUCKET_NAME}/{BLOB_NAME}")
    blob.download_to_filename(LOCAL_PATH)

    print(f"✅ Downloaded to {LOCAL_PATH}")

if __name__ == "__main__":
    download()


Writing download_sft_500.py


In [18]:
%%writefile sft_ddp.py
import os, json, math, random, requests, time, inspect
from dataclasses import dataclass
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import amp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

import tiktoken
from google.cloud import storage
import wandb

# -----------------------------
# Flash Attention v1 ON (T4 supports v1)
# -----------------------------
from torch.backends.cuda import sdp_kernel
sdp_kernel(enable_flash=True, enable_mem_efficient=False, enable_math=False)

# -----------------------------
# Config
# -----------------------------
DATA_URL  = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
DATA_PATH = "alpaca_data.json"

OUT_DIR = "sft_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_CKPT = "checkpoints/model_09535.pt"  # pretrained checkpoint
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# SFT Hyperparams (same as you had, only SEQ_LEN=1024)
BATCH_SIZE = 4
SEQ_LEN = 1024
GRAD_ACCUM_STEPS = 32
LR = 5e-5
WEIGHT_DECAY = 0.1
MAX_STEPS = 2000
WARMUP_STEPS = 100
EVAL_EVERY = 200
SAVE_EVERY = 500

SEED = 1337

# -----------------------------
# W&B setup (rank0 only)
# -----------------------------
WANDB_PROJECT = "gpt2-sft-ultrafineweb"
WANDB_RUN_NAME = "sft-ddp-1024"

# -----------------------------
# GCS setup
# -----------------------------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
BUCKET_NAME = "gpt2-ultrafineweb"
GCS_PREFIX = "sft_checkpoints"

def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(BUCKET_NAME)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"☁️ Uploaded: gs://{BUCKET_NAME}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# -----------------------------
# Import your model
# -----------------------------
from train_gpt2_v2 import GPT, GPTConfig  # your exact GPT class

enc = tiktoken.get_encoding("gpt2")
EOS_ID = enc.eot_token

# -----------------------------
# Dataset download
# -----------------------------
def download_alpaca():
    if os.path.exists(DATA_PATH):
        print(f"✅ Dataset already exists: {DATA_PATH}")
        return
    print("⬇️ Downloading Alpaca 52K...")
    r = requests.get(DATA_URL)
    r.raise_for_status()
    with open(DATA_PATH, "w") as f:
        f.write(r.text)
    print("✅ Downloaded Alpaca.")

def format_example(ex):
    inst = ex["instruction"].strip()
    inp = ex.get("input", "").strip()
    out = ex["output"].strip()

    if inp:
        text = (
            "### Instruction:\n"
            f"{inst}\n\n"
            "### Input:\n"
            f"{inp}\n\n"
            "### Response:\n"
            f"{out}"
        )
    else:
        text = (
            "### Instruction:\n"
            f"{inst}\n\n"
            "### Response:\n"
            f"{out}"
        )
    return text

def build_token_dataset_if_needed(master_process: bool):
    if os.path.exists("alpaca_train_tokens.pt") and os.path.exists("alpaca_val_tokens.pt"):
        if master_process:
            print("✅ Token files already exist.")
        return

    if master_process:
        download_alpaca()
        data = json.load(open(DATA_PATH))
        random.shuffle(data)

        val_size = 1000
        val_data = data[:val_size]
        train_data = data[val_size:]

        def tokenize_list(ex_list, desc):
            all_ids = []
            for ex in tqdm(ex_list, desc=desc):
                txt = format_example(ex)
                ids = enc.encode(txt)
                ids.append(EOS_ID)    # add actual token, NOT text
                all_ids.extend(ids)
            return torch.tensor(all_ids, dtype=torch.long)

        train_tokens = tokenize_list(train_data, "Tokenizing train")
        val_tokens   = tokenize_list(val_data, "Tokenizing val")

        torch.save(train_tokens, "alpaca_train_tokens.pt")
        torch.save(val_tokens, "alpaca_val_tokens.pt")
        print("✅ Saved alpaca_train_tokens.pt and alpaca_val_tokens.pt")

    # wait for rank0 to finish saving
    if dist.is_initialized():
        dist.barrier()

# -----------------------------
# DDP-aware TokenLoader
# each GPU reads different stride of token stream
# -----------------------------
class TokenLoaderDDP:
    def __init__(self, tokens, batch_size, seq_len, process_rank, num_processes):
        self.tokens = tokens
        self.B = batch_size
        self.T = seq_len
        self.rank = process_rank
        self.world = num_processes

        self.pos = self.B * self.T * self.rank

    def next_batch(self):
        B, T = self.B, self.T
        need = B*T + 1

        if self.pos + need >= len(self.tokens):
            self.pos = self.B * self.T * self.rank  # reset per-rank

        buf = self.tokens[self.pos : self.pos + need]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)

        self.pos += B*T*self.world
        return x, y

# -----------------------------
# Load pretrained model
# -----------------------------
def load_pretrained(ckpt_path, device):
    print(f"\n🔄 Loading pretrained ckpt: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config: GPTConfig = ckpt["config"]

    model = GPT(config)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    model.train()
    # model.half()

    print("✅ Pretrained model loaded.")
    return model, config

# -----------------------------
# LR schedule
# -----------------------------
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    return LR

# -----------------------------
# Eval loss
# -----------------------------
@torch.no_grad()
def eval_loss(model, val_loader, device, batches=30):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with amp.autocast("cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses)

# -----------------------------
# Sample generation
# -----------------------------
@torch.no_grad()
def sample_model(model, prompt, device, max_new=80, top_k=50):
    model.eval()
    ids = enc.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long)[None, :].to(device)

    for _ in range(max_new):
        logits, _ = model(x)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_idx = torch.topk(probs, k=top_k, dim=-1)
        ix = torch.multinomial(topk_probs, 1)
        next_id = torch.gather(topk_idx, -1, ix)
        x = torch.cat([x, next_id], dim=1)

    text = enc.decode(x[0].tolist())
    model.train()
    return text

# -----------------------------
# Main SFT (DDP)
# -----------------------------
def train_sft_ddp():
    # DDP init -------------------------------------------------
    ddp = int(os.environ.get("RANK", -1)) != -1
    if ddp:
        dist.init_process_group(backend="nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
        master_process = (rank == 0)
    else:
        rank, local_rank, world_size = 0, 0, 1
        device = DEVICE
        master_process = True

    # seeds ----------------------------------------------------
    random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    torch.cuda.manual_seed(SEED + rank)

    # build tokens once ---------------------------------------
    build_token_dataset_if_needed(master_process)

    train_tokens = torch.load("alpaca_train_tokens.pt")
    val_tokens   = torch.load("alpaca_val_tokens.pt")

    train_loader = TokenLoaderDDP(train_tokens, BATCH_SIZE, SEQ_LEN, rank, world_size)
    val_loader   = TokenLoaderDDP(val_tokens,   BATCH_SIZE, SEQ_LEN, rank, world_size)

    # model ----------------------------------------------------
    model, config = load_pretrained(BASE_CKPT, device)

    if ddp:
        model = DDP(model, device_ids=[local_rank])
    raw_model = model.module if ddp else model

    optimizer = raw_model.configure_optimizers(
        weight_decay=WEIGHT_DECAY,
        learning_rate=LR,
        device_type=device
    )
    scaler = amp.GradScaler(device="cuda")

    # wandb ----------------------------------------------------
    if master_process:
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config={
                "base_ckpt": BASE_CKPT,
                "batch_size": BATCH_SIZE,
                "seq_len": SEQ_LEN,
                "grad_accum_steps": GRAD_ACCUM_STEPS,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "max_steps": MAX_STEPS,
                "ddp_world_size": world_size,
                "flash_attention": True
            }
        )

    print("\n🚀 Starting SFT-DDP...")
    for step in range(MAX_STEPS):
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0

        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        for micro in range(GRAD_ACCUM_STEPS):
            x, y = train_loader.next_batch()
            x, y = x.to(device), y.to(device)

            with amp.autocast("cuda"):
                _, loss = model(x, y)

            loss = loss / GRAD_ACCUM_STEPS
            loss_accum += loss.item()

            if ddp:
                model.require_backward_grad_sync = (micro == GRAD_ACCUM_STEPS - 1)

            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()

        dt = time.time() - t0

        # average loss across GPUs
        if ddp:
            loss_t = torch.tensor(loss_accum, device=device)
            dist.all_reduce(loss_t, op=dist.ReduceOp.AVG)
            loss_accum = loss_t.item()

        if master_process and step % 10 == 0:
            print(f"step {step:4d} | train_loss {loss_accum:.4f} | lr {lr:.2e} | norm {norm:.2f} | dt {dt:.2f}s")
            wandb.log({
                "train/loss": loss_accum,
                "train/lr": lr,
                "train/grad_norm": norm,
                "step": step
            })

        # validation + sample (rank0 only)
        if master_process and step % EVAL_EVERY == 0 and step > 0:
            vloss = eval_loss(raw_model, val_loader, device)
            print(f"📉 val_loss @ step {step}: {vloss:.4f}")
            wandb.log({"val/loss": vloss, "step": step})

            prompt = "### Instruction:\nExplain what overfitting is in ML.\n\n### Response:\n"
            sample = sample_model(raw_model, prompt, device)
            print("\n🔮 sample:\n", sample[:1200], "\n")
            wandb.log({"sample": sample, "step": step})

        # save + upload every 500 steps (rank0 only)
        if master_process and step % SAVE_EVERY == 0 and step > 0:
            ckpt_path = os.path.join(OUT_DIR, f"sft_{step:05d}.pt")
            ckpt_obj = {
                "model": raw_model.state_dict(),
                "config": config,
                "step": step,
            }
            torch.save(ckpt_obj, ckpt_path)
            print(f"✅ saved {ckpt_path}")

            upload_to_gcs(ckpt_path, f"{GCS_PREFIX}/sft_{step:05d}.pt")
            wandb.save(ckpt_path)

    # final save (rank0 only)
    if master_process:
        final_path = os.path.join(OUT_DIR, "sft_final.pt")
        torch.save({
            "model": raw_model.state_dict(),
            "config": config,
            "step": MAX_STEPS,
        }, final_path)
        print(f"\n🏁 Finished! Final SFT checkpoint: {final_path}")
        upload_to_gcs(final_path, f"{GCS_PREFIX}/sft_final.pt")
        wandb.save(final_path)

    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    train_sft_ddp()


Overwriting sft_ddp.py


In [3]:
%%writefile train_gpt2_v2.py
# ON ULTRA-FINEWEB EDU DATASET 5B
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import inspect
import os
import math
from torch import amp
from torch.backends.cuda import sdp_kernel
sdp_kernel(enable_flash=False, enable_mem_efficient=True, enable_math=True)



# GOOGLE CLOUD UPLOAD SUPPORT
from google.cloud import storage

# Weights & Biases logging
import wandb
os.environ['WANDB_API_KEY'] = 'bc775a93d32c104af12794bd595056eb315f300f'
WANDB_KEY = os.getenv("WANDB_API_KEY")

wandb.login(key=WANDB_KEY)





# path to your JSON in kaggle/input
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gpt2-bucket-key.json"
os.environ["BUCKET_NAME"] = "gpt2-ultrafineweb"
bucket_name = "gpt2-ultrafineweb"
print("Service Account Key Loaded")

client = storage.Client()
bucket = client.bucket(bucket_name)

print("Auth OK, bucket exists:", bucket)


def upload_to_gcs(local_path, dest_blob_name):
    try:
        client = storage.Client()
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(dest_blob_name)
        blob.upload_from_filename(local_path)
        print(f"Uploaded to GCS: gs://{bucket_name}/{dest_blob_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

def download_latest_checkpoint(bucket_name, prefix="checkpoints", local_dir="checkpoints"):
    """
    Download only the latest checkpoint (.pt) file from GCS into local_dir.
    Skips download if same file already exists locally with the same size.
    Returns: local file path or None if not found.
    """
    os.makedirs(local_dir, exist_ok=True)

    client = storage.Client()
    bucket = client.bucket(bucket_name)

    # list blobs inside the prefix folder
    blobs = [b for b in bucket.list_blobs(prefix=prefix) if b.name.endswith(".pt")]

    if not blobs:
        print("❌ No checkpoints found in GCS.")
        return None

    # sort based on filename (model_00001.pt → latest is last)
    blobs.sort(key=lambda b: b.name)

    latest_blob = blobs[-1]
    filename = latest_blob.name.split("/")[-1]
    local_path = os.path.join(local_dir, filename)

    # If file already exists AND size matches, skip
    if os.path.exists(local_path) and os.path.getsize(local_path) == latest_blob.size:
        print(f"✅ Latest checkpoint already exists locally: {filename}")
        return local_path

    print(f"⬇️ Downloading latest checkpoint from GCS:\n  {latest_blob.name} → {local_path}")
    latest_blob.download_to_filename(local_path)

    print(f"✅ Download complete: {local_path}")
    return local_path

#--------------------------------------------------------
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd) #bias ON
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # not really a 'bias', more of a mask, but following the OpenAI/HF naming though
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
        #calculate query, key, values for all heads in batch and move head forward to be the batch
        # nh is "number of heads", hs is "head size", and C (number of channels) = nh * hs
        # e.g. in GPT-2 (124M), n_head=12, hs=64, so nh*hs=C=768 channels in the Transformer
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # attention (materializes the large (T,T) matrix for all the queries and keys)

        #flash attention v1 as T4 supports v1 only

        # att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        # att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        # att = F.softmax(att, dim=-1)
        # y = att @ v  # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True) # flash attention (inbuilt-pytorch version)


        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y



class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x




@dataclass
class GPTConfig:
    block_size: int = 1024 # max sequence length
    vocab_size: int = 50257 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token
    n_layer: int = 12 # number of layers
    n_head: int = 12 # number of heads
    n_embd: int = 768 # embedding dimension


class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight sharing scheme
        self.transformer.wte.weight = self.lm_head.weight

        # init params
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, "NANOGPT_SCALE_INIT"):
                std *= 2 * (self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # inside class GPT(nn.Module):

    def resize_embeddings(self, new_vocab_size: int):
      """Resize token embeddings and lm_head to new vocab size."""
      old_weight = self.transformer.wte.weight.data
      old_vocab_size, embed_dim = old_weight.shape

      new_embed = torch.zeros(new_vocab_size, embed_dim, device=old_weight.device)
      new_embed[:min(old_vocab_size, new_vocab_size)] = old_weight[:min(old_vocab_size, new_vocab_size)]

      self.transformer.wte = torch.nn.Embedding(new_vocab_size, embed_dim)
      self.transformer.wte.weight.data = new_embed

      self.lm_head = torch.nn.Linear(embed_dim, new_vocab_size, bias=False)
      self.lm_head.weight = self.transformer.wte.weight


    def forward(self, idx, targets=None):
        # idx is of shape (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size"
        # forward the token and position embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0) # shape (T)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (T, n_embd)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (B, T, n_embd)
        x = tok_emb + pos_emb
        # forward the blocks of the transformer
        for block in self.transformer.h:
            x = block(x)
        # forward the final layernorm and the classifer
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def configure_optimizers(self, weight_decay, learning_rate, device_type):
        # start with all of the candidate parameters (that require grad)
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        # if master_process:
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == "cuda"
        # if master_process:
        print(f"using fused AdamW: {use_fused}")
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
        return optimizer
# -----------------------------------------------------------------------------
import tiktoken
import numpy as np

def load_tokens(filename):
    npt = np.load(filename)
    ptt = torch.tensor(npt, dtype=torch.long)
    return ptt



class DataLoaderLite:
    def __init__(self, B, T, process_rank, num_processes, split):
        self.B = B
        self.T = T
        self.process_rank = process_rank
        self.num_processes = num_processes
        assert split in {'train', 'val'}

        #get the shard filenames
        data_root = "edu_ultrafineweb5B"
        shards = os.listdir(data_root)
        shards = [s for s in shards if split in s]
        shards = sorted(shards)
        shards = [os.path.join(data_root, s) for s in shards]
        self.shards = shards
        assert len(shards) > 0 , f"no shards found for split {split}"
        if master_process:
            print(f"found {len(shards)} shards for split {split}")

        # SHAKESPEAR DATASET:
        # # at init load tokens form disk and store them in memory
        # with open('input.txt', 'r') as f:
        #     text = f.read()
        # enc = tiktoken.get_encoding('gpt2')
        # tokens = enc.encode(text)
        # self.tokens = torch.tensor(tokens)
        # print(f"loaded {len(tokens)} tokens")
        # print(f"1 epoch = {len(self.tokens) // (B * T)} batches")

        #state, init at shard zero
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position+B*T+1]
        x = (buf[:-1]).view(B, T) # inputs
        y = (buf[1:]).view(B, T) # outputs
        # advance the position in the tensor
        self.current_position += B * T * self.num_processes
        # if loading the next batch would be out of bounds, reset
        if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards)
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.current_position = B * T * self.process_rank
        return x, y


def load_checkpoint(checkpoint_path, model, optimizer, scaler):
    """Load checkpoint and restore training state"""
    print(f"Loading checkpoint from {checkpoint_path}")

    import torch.serialization
    torch.serialization.add_safe_globals([GPTConfig])

    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scaler.load_state_dict(checkpoint['scaler'])

    # Restore RNG states
    torch.set_rng_state(checkpoint['rng_state'])
    if checkpoint['cuda_rng_state'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state(checkpoint['cuda_rng_state'])

    start_step = checkpoint['step'] + 1
    print(f"Resumed from step {checkpoint['step']}, starting at step {start_step}")
    return start_step




@torch.no_grad()
def evaluate_val_loss(model, val_loader, batches=20):
    """Run on a few batches of val set and compute loss + perplexity."""
    model.eval()

    losses = []
    for _ in range(batches):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)

        with amp.autocast("cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())

    avg_loss = sum(losses) / len(losses)
    ppl = math.exp(avg_loss)

    model.train()
    return avg_loss, ppl

if __name__ == "__main__":
    pass
# # -----------------------------------------------------------------------------

#     # simple launch:
#     # python train_gpt2.py
#     # DDP launch for e.g. 8 GPUs:
#     # torchrun --standalone --nproc_per_node=8 train_gpt2.py

#     # attempt to autodetect the device

#     os.environ.setdefault("NCCL_P2P_DISABLE", "1")
#     os.environ.setdefault("NCCL_IB_DISABLE", "1")
#     os.environ.setdefault("NCCL_ASYNC_ERROR_HANDLING", "1")

#     import time
#     torch.backends.cuda.matmul.allow_tf32 = True  # no effect on T4, safe to enable
#     torch.backends.cudnn.allow_tf32 = True
#     torch.set_float32_matmul_precision("high")


#     # run the training loop
#     from torch.distributed import init_process_group, destroy_process_group
#     from torch.nn.parallel import DistributedDataParallel as DDP
#     import torch.distributed as dist



#     device = "cpu"
#     if torch.cuda.is_available():
#         device = "cuda"
#     elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
#         device = "mps"
#     print(f"using device: {device}")
#     # device = "cpu" #OVERRIDE
#     max_length = 30
#     num_return_sequences = 5

#     log_dir = "checkpoints"
#     os.makedirs(log_dir, exist_ok=True)
#     log_file = os.path.join(log_dir, f"log.txt")

#     # set up DDP (distributed data parallel).
#     # torchrun command sets the env variables RANK, LOCAL_RANK, and WORLD_SIZE
#     ddp = int(os.environ.get('RANK', -1)) != -1 # is this a ddp run?
#     if ddp:
#         # use of DDP atm demands CUDA, we set the device appropriately according to rank
#         assert torch.cuda.is_available(), "for now i think we need CUDA for DDP"
#         init_process_group(backend='nccl')
#         ddp_rank = int(os.environ['RANK'])
#         ddp_local_rank = int(os.environ['LOCAL_RANK'])
#         ddp_world_size = int(os.environ['WORLD_SIZE'])
#         device = f'cuda:{ddp_local_rank}'
#         torch.cuda.set_device(device)
#         master_process = ddp_rank == 0 # this process will do logging, checkpointing etc.
#     else:
#         # vanilla, non-DDP run
#         ddp_rank = 0
#         ddp_local_rank = 0
#         ddp_world_size = 1
#         master_process = True
#         # attempt to autodetect device
#         device = "cpu"
#         if torch.cuda.is_available():
#             device = "cuda"
#         elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
#             device = "mps"
#         print(f"using device: {device}")

#     device_type = "cuda" if device.startswith("cuda") else "cpu"

#     torch.manual_seed(1337)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(1337)

#     total_batch_size = 524288 # 2**19, ~0.5M, in number of tokens
#     B = 8 # micro batch size
#     T = 1024 # sequence length
#     assert total_batch_size % (B * T * ddp_world_size) == 0, "make sure total_batch_size is divisible by B * T"
#     grad_accum_steps = total_batch_size // (B * T * ddp_world_size)
#     if master_process:
#         print(f"total desired batch size: {total_batch_size}")
#         print(f"calculated gradient accumulation steps: {grad_accum_steps}")

#         wandb.init(
#             project="gpt2-ultra-fineweb",
#             name="kaggle-ddp-training",
#             config={
#                 "model": "GPT2_124M_from_scratch",
#                 "batch_size": 8,
#                 "seq_len": 1024,
#                 "grad_accum_steps": grad_accum_steps,
#                 "dataset": "UltraFineWeb-5B",
#                 "precision": "fp16 + GradScaler",
#             }
#         )


#     train_loader = DataLoaderLite(B=B, T=T, process_rank=ddp_rank, num_processes=ddp_world_size, split="train")

#     val_loader = DataLoaderLite(B=B, T=T, process_rank=ddp_rank, num_processes=ddp_world_size, split="val")

#     # create model
#     model = GPT(GPTConfig(vocab_size=50304))
#     # model = GPT.from_pretrained("gpt2")
#     model.to(device)
#     #compile the model
#     # model = torch.compile(model)
#     if ddp:
#         model = DDP(model, device_ids=[ddp_local_rank])
#     raw_model = model.module if ddp else model # always contains the "raw" unwrapped model

#     max_lr = 6e-4
#     min_lr = max_lr * 0.1
#     warmup_steps = 715
#     max_steps = 9536 # as 5B tokens (5e9 / 2**19)
#     def get_lr(it):
#         # 1) linear warmup for warmup_iters steps
#         if it < warmup_steps:
#             return max_lr * (it+1) / warmup_steps
#         # 2) if it > lr_decay_iters, return min learning rate
#         if it >= max_steps:
#             return min_lr
#         # 3) in between, use consine decay down to min learning rate
#         decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
#         assert 0 <= decay_ratio <= 1
#         coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff ranges at 1 and goes to 0
#         return min_lr + coeff * (max_lr - min_lr)

#     #optimize!

#     scaler = amp.GradScaler(device="cuda")  # NEW: enable mixed precision

#     optimizer = raw_model.configure_optimizers(weight_decay=0.1, learning_rate=6e-4, device_type=device)
#     # optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), eps=1e-8)

#     start_step = 0

#     if master_process:  # only rank0 scans filesystem

#         latest_gcs_ckpt = None
#         ckpts = [f for f in os.listdir(log_dir) if f.endswith(".pt")]
#         if not ckpts:
#             print("No local checkpoint found — checking Google Cloud...")
#             latest_gcs_ckpt = download_latest_checkpoint(bucket_name="gpt2-ultrafineweb")

#             if latest_gcs_ckpt:
#                 print(f"Loaded checkpoint from GCS: {latest_gcs_ckpt}")
#                 ckpts = [latest_gcs_ckpt]   # treat as local checkpoint now

#         if ckpts:
#             ckpts.sort()  # ensures last checkpoint is latest
#             latest_ckpt = ckpts[-1]
#             if not latest_ckpt.startswith(log_dir):
#                 latest_ckpt = os.path.join(log_dir, latest_ckpt)

#             print(f"▶ Found checkpoint: {latest_ckpt}")
#             start_step = load_checkpoint(latest_ckpt, raw_model, optimizer, scaler)
#             ckpt = torch.load(latest_ckpt, map_location="cpu")
#             if "data_state" in ckpt:
#                 train_loader.current_shard = ckpt["data_state"]["current_shard"]
#                 train_loader.tokens = load_tokens(train_loader.shards[train_loader.current_shard])
#                 train_loader.current_position = ckpt["data_state"]["current_position"]

#             avg_step_time = ckpt.get("avg_step_time", None)
#         else:
#             print("▶ No checkpoint found — starting fresh.")

#     # Sync start_step across all GPUs if running DDP
#     if ddp:
#         start_step_tensor = torch.tensor([start_step], device=device)
#         dist.broadcast(start_step_tensor, src=0)
#         start_step = int(start_step_tensor.item())

#     if master_process:
#         print(f"Training will start at step = {start_step} / {max_steps}")


#     for step in range(start_step, max_steps):

#         t0 = time.time()
#         optimizer.zero_grad()
#         loss_accum = 0.0

#         #determine and set the learning rate for this iteration
#         lr = get_lr(step)
#         for param_group in optimizer.param_groups:
#             param_group['lr'] = lr

#         for micro_step in range(grad_accum_steps):
#             x, y = train_loader.next_batch()
#             x, y = x.to(device), y.to(device)

#             # FP16 forward + loss (autocast enables mixed precision on T4)
#             with amp.autocast("cuda"):
#                 logits, loss = model(x, y)

#             # we have to scale the loss to account for gradient accumulation,
#             # because the gradients just add on each successive backward().
#             # addition of gradients corresponds to a SUM in the objective, but
#             # instead of a SUM we want MEAN. Scale the loss here so it comes out right.
#             loss = loss / grad_accum_steps # scale the loss
#             loss_accum += loss.item()

#             if ddp:
#                 model.require_backward_grad_sync = (micro_step == grad_accum_steps - 1)
#             # backward pass (scaled for safe gradients in FP16)
#             scaler.scale(loss).backward()
#         if ddp:
#             loss_accum_tensor = torch.tensor(loss_accum, dtype=torch.float32, device=device)
#             dist.all_reduce(loss_accum_tensor, op=dist.ReduceOp.AVG)
#             loss_accum = loss_accum_tensor.item()
#         #unscale before clipping
#         scaler.unscale_(optimizer)
#         norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)


#         # update weights
#         scaler.step(optimizer)
#         scaler.update()

#         torch.cuda.synchronize() # wait for GPU to finish work
#         t1 = time.time()

#         dt = (t1 - t0) # time difference
#         ms = dt * 1000 #ms
#         tokens_processed = train_loader.B * train_loader.T * grad_accum_steps * ddp_world_size
#         tokens_per_sec = tokens_processed / dt
#         if step == start_step:
#             avg_step_time = dt
#         else:
#             avg_step_time = avg_step_time * 0.9 + dt * 0.1  # EMA smoothing

#         steps_left = max_steps - step - 1
#         remaining_sec = steps_left * avg_step_time

#         hrs = int(remaining_sec // 3600)
#         mins = int((remaining_sec % 3600) //#scrollTo=qHmihQqmdyAl 60)
#         if master_process:

#             wandb.log({
#                 "train/loss": loss_accum,
#                 "train/lr": lr,
#                 "train/grad_norm": norm,
#                 "train/tokens_per_sec": tokens_per_sec,
#                 "train/step_time_ms": ms,
#                 "step": step,
#             })

#             print(
#                 f" step {step:4d} | loss: {loss_accum:.6f} | lr: {lr:.4e} | "
#                 f"norm: {norm:.4f} | dt: {ms:.2f}ms | tok/sec: {tokens_per_sec:.2f} | "
#                 f"ETA: {hrs:02d}h:{mins:02d}m"
#             )

#             # --- VALIDATION CHECK ---
#             if step % 200 == 0:   # adjust frequency as needed
#                 val_loss, val_ppl = evaluate_val_loss(raw_model, val_loader)
#                 wandb.log({
#                     "val/loss": val_loss,
#                     "val/perplexity": val_ppl,
#                 })
#                 print(f" VAL | loss: {val_loss:.6f} | ppl: {val_ppl:.2f}")


#             if step % 500 == 0:  # change to whatever frequency you want
#                     raw_model.eval()  # disable dropout/layernorm noise

#                     prompt = "Hello, I'm training a GPT model,"
#                     enc = tiktoken.get_encoding("gpt2")
#                     tokens = enc.encode(prompt)
#                     tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).repeat(num_return_sequences, 1).to(device)

#                     max_gen_len = 40  # maximum generated length

#                     with torch.no_grad():
#                         for _ in range(max_gen_len - tokens.shape[1]):
#                             logits, _ = raw_model(tokens)
#                             logits = logits[:, -1, :]
#                             probs = F.softmax(logits, dim=-1)
#                             topk_probs, topk_indices = torch.topk(probs, k=50, dim=-1)
#                             ix = torch.multinomial(topk_probs, 1)
#                             xcol = torch.gather(topk_indices, -1, ix)
#                             tokens = torch.cat((tokens, xcol), dim=1)

#                     print("\n🔮 SAMPLE GENERATION (step", step, ")")
#                     for i in range(num_return_sequences):
#                         decoded = enc.decode(tokens[i].tolist())
#                         print(f"   > {decoded}")
#                         wandb.log({f"sample_{i}": decoded})

#                     raw_model.train()  # switch model back to train mode

#             # Save checkpoint every 10 steps (adjust as needed)
#             # if step == max_steps - 1:  #for final
#             # if step > 0 and (step % 200 == 0 or step == max_steps - 1): #for after every 10 epochs
#             if step > 0 and (step % 500 == 0 or step == max_steps - 1): #for after every 10 epochs
#                 checkpoint_path = os.path.join(log_dir, f"model_{step:05d}.pt")
#                 checkpoint = {
#                     'model': raw_model.state_dict(),
#                     'config': raw_model.config,
#                     'step': step,
#                     'train_loss': loss_accum,
#                     'optimizer': optimizer.state_dict(),
#                     'scaler': scaler.state_dict(),  # save scaler state for mixed precision
#                     'rng_state': torch.get_rng_state(),
#                     'cuda_rng_state': torch.cuda.get_rng_state() if torch.cuda.is_available() else None,
#                     'data_state': {
#                         'current_shard': train_loader.current_shard,
#                         'current_position': train_loader.current_position,
#                     },
#                     'val_loss': val_loss if 'val_loss' in locals() else None,
#                     'val_perplexity': val_ppl if 'val_ppl' in locals() else None,

#                 }
#                 checkpoint['avg_step_time'] = avg_step_time if 'avg_step_time' in locals() else None
#                 torch.save(checkpoint, checkpoint_path)
#                 upload_to_gcs(checkpoint_path, f"checkpoints/model_{step:05d}.pt")
#                 wandb.save(checkpoint_path)


#                 print(f"Checkpoint saved at step {step}: {checkpoint_path}")
#                 # After torch.save(...)
#                 keep = 3
#                 ckpts = sorted([f for f in os.listdir(log_dir) if f.endswith(".pt")])
#                 while len(ckpts) > keep:
#                     old = os.path.join(log_dir, ckpts.pop(0))
#                     try:
#                         os.remove(old)
#                         print(f"Removed old checkpoint: {old}")
#                     except Exception as e:
#                         print(f"Warn: could not remove {old}: {e}")


#     if ddp:
#         destroy_process_group()

#     import sys; sys.exit(0)


#     #prefix tokens
#     import tiktoken
#     enc = tiktoken.get_encoding("gpt2")
#     tokens = enc.encode("Hello, I'm a language model,")
#     tokens = torch.tensor(tokens, dtype=torch.long) #(8, )
#     tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1) #(5, 8)
#     x = tokens.to(device)


#     # generate! right now x is (B, T) where B = 5, T = 8
#     # set the seed is 42 (my fav what is life :P)
#     torch.manual_seed(42)
#     torch.cuda.manual_seed(42)
#     while x.size(1) < max_length:
#         # forward the model to get the logits
#         with torch.no_grad():
#             logits = model(x) # (B, T, vocab_size)
#             # take the logits at the last position
#             logits = logits[:, -1, :] # (B, vocab_size)
#             # get the probabilities
#             probs = F.softmax(logits, dim=-1)
#             # do top-k sampling of 50 (huggingface pipeline default)
#             # topk_probs here becomes (5, 50), topk_indices is  (5, 50)
#             topk_probs, topk_indices = torch.topk(probs, k=50, dim=-1)
#             # select a token from the top-k probabilities
#             ix = torch.multinomial(topk_probs, num_samples=1) # (B, 1)
#             # gather the corresponding indices
#             xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
#             # append to the sequnce
#             x = torch.cat((x, xcol), dim=1)

#     # print the generated text
#     for i in range(num_return_sequences):
#         tokens = x[i, :max_length].tolist()
#         decoded = enc.decode(tokens)
#         print(">", decoded)



Writing train_gpt2_v2.py
